# Dibujo de soportes por clics: revisión de la interfaz

Notebook nuevo (no modifica el ejemplo `agente_estructural.ipynb`). Reúne lo avanzado:

- **Dibujo directo por clics** (estilo RAM/SAP): planta X-Y y alzado X-Z, nodos por clic, barras entre nodos, apoyos y rejilla con ajuste.
- **Base de perfiles Gerdau México rev04 incrustada** (247 perfiles, no ICHA). No hay que subir archivos.
- **Asignación de perfil por lista desplegable**, ordenada por familia (IR, CE, LI, SOL, OS, CS), peralte y masa. Muestra la masa en kg/m y el estado del dato si no es Revisado.
- **Orientación del eje fuerte (peralte)**: vigas con peralte vertical por defecto, columnas con peralte en Y global.
- Salida `dib.a_modelo()` con el mismo formato que `leer_dxf()`, para conectar luego con el motor de análisis.

## Índice de bloques

1. Modo interactivo de la figura
2. Motor de dibujo: nodos, barras, apoyos (dibujo_directo.py)
3. Datos Gerdau rev04 incrustados (gerdau_datos.py)
4. Base de perfiles, orden y lista desplegable (perfiles_gerdau.py)
5. Clics y modos de dibujo (dibujo_clics.py)
6. Orientacion del eje fuerte / peralte (orientacion_gerdau.py)
7. Interfaz de clics con planta, alzado y seccion (interfaz_clics.py)
8. Demo: marco, apoyos, orientacion, lista de perfiles y pantalla

**Cómo se usa:** ejecutar los bloques en orden, del 1 al 8 (los bloques 2 a 7 solo escriben archivos; el 8 es la demo). Elegir el perfil en la lista desplegable que aparece encima de la figura. Los botones de la izquierda cambian el modo: Seleccionar, Nodo, Barra, Poli, Apoyo, Eje, Borrar. Clic derecho cancela. Planta/Alzado cambia la vista.

**Limitaciones de esta versión:** la selección es en vista 2D (planta y alzado); la selección 3D no está. Solo se modela apoyo empotrado. El cálculo no está en este notebook. La lista desplegable no tiene encabezados por familia (no lo soporta el control); el orden es el que separa las familias.


## Bloque 1. Modo interactivo de la figura

Cargar el modo interactivo de la figura.

In [ ]:
# BLOQUE 1: modo interactivo de la figura (ipympl ya viene instalado en Kaggle)
%matplotlib widget


## Bloque 2. Motor de dibujo: nodos, barras, apoyos (dibujo_directo.py)

Nodos (con tolerancia), barras por elemento y perfil, apoyos, ejes fuertes y marco por ejes.

In [ ]:
%%writefile dibujo_directo.py
"""
MODULO DE DIBUJO DIRECTO (reemplaza la lectura de DXF).

Permite construir el modelo estructural dentro del notebook: nodos (puntos x, y, z en m),
barras (polilineas entre nodos), apoyos y ejes fuertes. Produce EXACTAMENTE el mismo
diccionario que entrega lector_validador.leer_dxf(), para que el motor de analisis
(motor_analisis.construir_y_resolver) y el validador funcionen sin cambios.

Formato de salida (igual al DXF):
  {'NODOS': [(x,y,z) en m], 'BARRAS': [{'n1','n2','elemento','perfil','capa','eje_fuerte'}],
   'NODOS_APOYO': [indices], 'ruta': str}

Unidades: metros (la conversion a cm la hace el motor).
"""
import json
import numpy as np

TOL_NODO = 1e-3                 # m: nodos a menos de esta distancia son el mismo nodo
ELEMENTOS = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
TIPOS_APOYO = ['empotrado']     # el motor actual solo modela empotrado (6 GDL); rotulado/rodillo pendiente


class DibujoModelo:
    def __init__(self, tol=TOL_NODO, ruta='dibujo_directo'):
        self.nodos = []         # lista de tuplas (x, y, z)
        self.barras = []        # lista de dicts con el mismo formato que el DXF
        self.apoyos = {}        # {indice_nodo: tipo}
        self.tol = tol
        self.ruta = ruta

    # ---------- nodos ----------
    def nodo(self, x, y, z):
        """Agrega un nodo (o devuelve el existente a menos de la tolerancia). Devuelve su indice."""
        p = (round(float(x), 4), round(float(y), 4), round(float(z), 4))
        for i, q in enumerate(self.nodos):
            if all(abs(p[k] - q[k]) < self.tol for k in range(3)):
                return i
        self.nodos.append(p)
        return len(self.nodos) - 1

    # ---------- barras ----------
    def _existe(self, n1, n2):
        return any({b['n1'], b['n2']} == {n1, n2} for b in self.barras)

    def barra(self, n1, n2, elemento, perfil):
        """Agrega una barra entre dos nodos (indices). Si ya existe, no la duplica. Devuelve su indice o None."""
        if n1 == n2:
            raise ValueError(f'barra degenerada en el nodo {n1}')
        if self._existe(n1, n2):
            return None
        elemento = str(elemento).upper()
        if elemento not in ELEMENTOS:
            raise ValueError(f'elemento {elemento!r} no reconocido; usar {ELEMENTOS}')
        self.barras.append({'n1': n1, 'n2': n2, 'elemento': elemento, 'perfil': perfil,
                            'capa': f'{elemento}-{perfil}', 'eje_fuerte': None})
        return len(self.barras) - 1

    def polilinea(self, puntos, elemento, perfil):
        """Une una secuencia de puntos con barras consecutivas (estilo 'dibujar con lineas').
        puntos: lista de indices de nodo o de coordenadas (x, y, z). Devuelve indices de barras creadas."""
        ids = [p if isinstance(p, (int, np.integer)) else self.nodo(*p) for p in puntos]
        creadas = []
        for a, b in zip(ids[:-1], ids[1:]):
            r = self.barra(a, b, elemento, perfil)
            if r is not None:
                creadas.append(r)
        return creadas

    def eje_fuerte(self, indice_barra, direccion):
        """Fija la direccion del eje fuerte de una barra (vector en coordenadas globales)."""
        d = np.array(direccion, float)
        n = np.linalg.norm(d)
        if n < 1e-9:
            raise ValueError('direccion nula')
        self.barras[indice_barra]['eje_fuerte'] = tuple(float(v) for v in d / n)

    # ---------- apoyos ----------
    def apoyo(self, nodo_idx, tipo='empotrado'):
        if tipo not in TIPOS_APOYO:
            raise ValueError(f'tipo de apoyo {tipo!r} no soportado por el motor (usar {TIPOS_APOYO})')
        self.apoyos[int(nodo_idx)] = tipo

    # ---------- generadores ----------
    def marco_grilla(self, xs, ys, zs, perfil_columna, perfil_viga_x, perfil_viga_y, eje_fuerte_columna=None):
        """Marco ortogonal a partir de ejes: nodos en todas las intersecciones (x, y, z).
        Columnas en cada (x, y) entre niveles consecutivos; vigas X y vigas Y en cada nivel superior.
        eje_fuerte_columna: direccion del eje fuerte para las columnas (opcional, ej. (1,0,0))."""
        xs, ys, zs = sorted(xs), sorted(ys), sorted(zs)
        idx = {}
        for x in xs:
            for y in ys:
                for z in zs:
                    idx[(x, y, z)] = self.nodo(x, y, z)
        for x in xs:
            for y in ys:
                for za, zb in zip(zs[:-1], zs[1:]):
                    i = self.barra(idx[(x, y, za)], idx[(x, y, zb)], 'COLUMNA', perfil_columna)
                    if i is not None and eje_fuerte_columna is not None:
                        self.eje_fuerte(i, eje_fuerte_columna)
        for y in ys:
            for z in zs[1:]:
                for xa, xb in zip(xs[:-1], xs[1:]):
                    self.barra(idx[(xa, y, z)], idx[(xb, y, z)], 'VIGA', perfil_viga_x)
        for x in xs:
            for z in zs[1:]:
                for ya, yb in zip(ys[:-1], ys[1:]):
                    self.barra(idx[(x, ya, z)], idx[(x, yb, z)], 'VIGA', perfil_viga_y)
        return idx

    # ---------- salida y persistencia ----------
    def a_modelo(self):
        """Diccionario con el formato de leer_dxf(): listo para motor_analisis y validar_modelo."""
        return {'NODOS': list(self.nodos), 'BARRAS': [dict(b) for b in self.barras],
                'NODOS_APOYO': sorted(self.apoyos.keys()), 'ruta': self.ruta}

    def guardar_json(self, ruta):
        datos = {'nodos': self.nodos, 'barras': self.barras,
                 'apoyos': {str(k): v for k, v in self.apoyos.items()}}
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=1)

    @classmethod
    def cargar_json(cls, ruta):
        with open(ruta, encoding='utf-8') as f:
            datos = json.load(f)
        m = cls()
        m.nodos = [tuple(p) for p in datos['nodos']]
        m.barras = datos['barras']
        m.apoyos = {int(k): v for k, v in datos['apoyos'].items()}
        return m


def comparar_modelos(m1, m2, tol=1e-3):
    """Compara dos modelos (formato leer_dxf) sin depender del orden de nodos ni barras.
    Devuelve (ok, lista_de_diferencias)."""
    def claves_nodos(m):
        return {tuple(round(c, 3) for c in p) for p in m['NODOS']}

    def claves_barras(m):
        out = []
        for b in m['BARRAS']:
            p1 = tuple(round(c, 3) for c in m['NODOS'][b['n1']])
            p2 = tuple(round(c, 3) for c in m['NODOS'][b['n2']])
            ext = tuple(sorted([p1, p2]))
            ef = None if b['eje_fuerte'] is None else tuple(round(c, 3) for c in b['eje_fuerte'])
            out.append((ext, b['elemento'], b['perfil'], ef))
        return sorted(out, key=str)

    def claves_apoyos(m):
        return sorted(tuple(round(c, 3) for c in m['NODOS'][i]) for i in m['NODOS_APOYO'])

    dif = []
    if claves_nodos(m1) != claves_nodos(m2):
        dif.append(f"nodos distintos: {len(claves_nodos(m1))} vs {len(claves_nodos(m2))}")
    b1, b2 = claves_barras(m1), claves_barras(m2)
    if b1 != b2:
        s1, s2 = set(map(str, b1)), set(map(str, b2))
        dif.append(f"barras distintas: {len(b1)} vs {len(b2)} | solo en 1: {len(s1 - s2)} | solo en 2: {len(s2 - s1)}")
    if claves_apoyos(m1) != claves_apoyos(m2):
        dif.append("apoyos distintos")
    return (len(dif) == 0), dif


## Bloque 3. Datos Gerdau rev04 incrustados (gerdau_datos.py)

Los 247 perfiles en un solo texto. No hay que editar este bloque.

In [ ]:
%%writefile gerdau_datos.py
_DATOS_TEXTO = "[{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 8.5\",\"estado\":\"Revisado\",\"masa_kg_m\":12.7,\"d_ala_diametro_cm\":14.8,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.49000000000000005,\"area_cm2\":16.3,\"ix_cm4\":620,\"iy_cm4\":82,\"sx_cm3\":83.78378378378378,\"sy_cm3\":16.4,\"rx_cm\":6.2,\"ry_cm\":2.2,\"j_cm4\":1.4,\"cw_cm6\":4222.3},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 9\",\"estado\":\"Revisado\",\"masa_kg_m\":13.6,\"d_ala_diametro_cm\":15,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.55,\"area_cm2\":17.3,\"ix_cm4\":683,\"iy_cm4\":91,\"sx_cm3\":91.06666666666666,\"sy_cm3\":18.2,\"rx_cm\":6.3,\"ry_cm\":2.3,\"j_cm4\":1.7,\"cw_cm6\":4750.26},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 12\",\"estado\":\"Con restricciones\",\"masa_kg_m\":18,\"d_ala_diametro_cm\":15.3,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.71,\"area_cm2\":22.9,\"ix_cm4\":920,\"iy_cm4\":124,\"sx_cm3\":120.26143790849673,\"sy_cm3\":24.31372549019608,\"rx_cm\":6.3,\"ry_cm\":2.3,\"j_cm4\":3.7,\"cw_cm6\":6598.91},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 16\",\"estado\":\"Revisado\",\"masa_kg_m\":24,\"d_ala_diametro_cm\":16,\"bf_b_cm\":10.2,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.03,\"area_cm2\":30.6,\"ix_cm4\":1336,\"iy_cm4\":184,\"sx_cm3\":167,\"sy_cm3\":36.07843137254902,\"rx_cm\":6.6,\"ry_cm\":2.5,\"j_cm4\":9.2,\"cw_cm6\":10308.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 15\",\"estado\":\"Revisado\",\"masa_kg_m\":22.4,\"d_ala_diametro_cm\":15.2,\"bf_b_cm\":15.2,\"tw_cm\":0.58,\"tf_t_cm\":0.6599999999999999,\"area_cm2\":28.6,\"ix_cm4\":1211,\"iy_cm4\":388,\"sx_cm3\":159.3421052631579,\"sy_cm3\":51.05263157894737,\"rx_cm\":6.5,\"ry_cm\":3.7,\"j_cm4\":4.2,\"cw_cm6\":20506.93},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 20\",\"estado\":\"Revisado\",\"masa_kg_m\":29.7,\"d_ala_diametro_cm\":15.7,\"bf_b_cm\":15.3,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":0.93,\"area_cm2\":37.9,\"ix_cm4\":1723,\"iy_cm4\":554,\"sx_cm3\":219.4904458598726,\"sy_cm3\":72.41830065359477,\"rx_cm\":6.8,\"ry_cm\":3.8,\"j_cm4\":9.9,\"cw_cm6\":30214.18},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 25\",\"estado\":\"Revisado\",\"masa_kg_m\":37.2,\"d_ala_diametro_cm\":16.2,\"bf_b_cm\":15.4,\"tw_cm\":0.8099999999999999,\"tf_t_cm\":1.16,\"area_cm2\":47.4,\"ix_cm4\":2223,\"iy_cm4\":712,\"sx_cm3\":274.44444444444446,\"sy_cm3\":92.46753246753246,\"rx_cm\":6.9,\"ry_cm\":3.9,\"j_cm4\":19.1,\"cw_cm6\":40263.88},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 10\",\"estado\":\"Revisado\",\"masa_kg_m\":15,\"d_ala_diametro_cm\":20,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.52,\"area_cm2\":19.1,\"ix_cm4\":1282,\"iy_cm4\":87,\"sx_cm3\":128.2,\"sy_cm3\":17.4,\"rx_cm\":8.2,\"ry_cm\":2.1,\"j_cm4\":1.7,\"cw_cm6\":8253.48},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 13\",\"estado\":\"Revisado\",\"masa_kg_m\":19.4,\"d_ala_diametro_cm\":20.3,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.65,\"area_cm2\":24.8,\"ix_cm4\":1648,\"iy_cm4\":114,\"sx_cm3\":162.36453201970443,\"sy_cm3\":22.35294117647059,\"rx_cm\":8.2,\"ry_cm\":2.1,\"j_cm4\":3.7,\"cw_cm6\":11004.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 15\",\"estado\":\"Revisado\",\"masa_kg_m\":22.5,\"d_ala_diametro_cm\":20.6,\"bf_b_cm\":10.2,\"tw_cm\":0.62,\"tf_t_cm\":0.8,\"area_cm2\":28.6,\"ix_cm4\":1998,\"iy_cm4\":142,\"sx_cm3\":193.98058252427182,\"sy_cm3\":27.84313725490196,\"rx_cm\":8.4,\"ry_cm\":2.2,\"j_cm4\":5.8,\"cw_cm6\":13917.42},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 18\",\"estado\":\"Revisado\",\"masa_kg_m\":26.6,\"d_ala_diametro_cm\":20.7,\"bf_b_cm\":13.3,\"tw_cm\":0.58,\"tf_t_cm\":0.8400000000000001,\"area_cm2\":33.9,\"ix_cm4\":2576,\"iy_cm4\":332,\"sx_cm3\":248.88888888888889,\"sy_cm3\":49.92481203007519,\"rx_cm\":8.7,\"ry_cm\":3.1,\"j_cm4\":7.1,\"cw_cm6\":32736.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 21\",\"estado\":\"Revisado\",\"masa_kg_m\":31.2,\"d_ala_diametro_cm\":21,\"bf_b_cm\":13.4,\"tw_cm\":0.64,\"tf_t_cm\":1.02,\"area_cm2\":39.7,\"ix_cm4\":3134,\"iy_cm4\":407,\"sx_cm3\":298.4761904761905,\"sy_cm3\":60.74626865671642,\"rx_cm\":8.9,\"ry_cm\":3.2,\"j_cm4\":11.7,\"cw_cm6\":40618.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 24\",\"estado\":\"Revisado\",\"masa_kg_m\":35.9,\"d_ala_diametro_cm\":20.1,\"bf_b_cm\":16.5,\"tw_cm\":0.62,\"tf_t_cm\":1.02,\"area_cm2\":45.7,\"ix_cm4\":3446,\"iy_cm4\":762,\"sx_cm3\":342.88557213930346,\"sy_cm3\":92.36363636363636,\"rx_cm\":8.7,\"ry_cm\":4.1,\"j_cm4\":14.6,\"cw_cm6\":69350.84},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 28\",\"estado\":\"Revisado\",\"masa_kg_m\":41.8,\"d_ala_diametro_cm\":20.5,\"bf_b_cm\":16.6,\"tw_cm\":0.72,\"tf_t_cm\":1.1800000000000002,\"area_cm2\":53.2,\"ix_cm4\":4079,\"iy_cm4\":903,\"sx_cm3\":397.9512195121951,\"sy_cm3\":108.79518072289156,\"rx_cm\":8.8,\"ry_cm\":4.1,\"j_cm4\":22.5,\"cw_cm6\":84263.99},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 31\",\"estado\":\"Revisado\",\"masa_kg_m\":46.2,\"d_ala_diametro_cm\":20.3,\"bf_b_cm\":20.3,\"tw_cm\":0.72,\"tf_t_cm\":1.1,\"area_cm2\":58.9,\"ix_cm4\":4579,\"iy_cm4\":1544,\"sx_cm3\":451.13300492610836,\"sy_cm3\":152.11822660098522,\"rx_cm\":8.8,\"ry_cm\":5.1,\"j_cm4\":22.5,\"cw_cm6\":142295.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":20.6,\"bf_b_cm\":20.4,\"tw_cm\":0.79,\"tf_t_cm\":1.26,\"area_cm2\":66.5,\"ix_cm4\":5286,\"iy_cm4\":1773,\"sx_cm3\":513.2038834951456,\"sy_cm3\":173.82352941176472,\"rx_cm\":8.9,\"ry_cm\":5.2,\"j_cm4\":32,\"cw_cm6\":165791.28},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.3,\"d_ala_diametro_cm\":21,\"bf_b_cm\":20.5,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.42,\"area_cm2\":75.5,\"ix_cm4\":6077,\"iy_cm4\":2044,\"sx_cm3\":578.7619047619048,\"sy_cm3\":199.41463414634146,\"rx_cm\":9,\"ry_cm\":5.2,\"j_cm4\":46.6,\"cw_cm6\":195905.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 48\",\"estado\":\"Revisado\",\"masa_kg_m\":71.4,\"d_ala_diametro_cm\":21.6,\"bf_b_cm\":20.6,\"tw_cm\":1.02,\"tf_t_cm\":1.7399999999999998,\"area_cm2\":91,\"ix_cm4\":7659,\"iy_cm4\":2535,\"sx_cm3\":709.1666666666666,\"sy_cm3\":246.1165048543689,\"rx_cm\":9.2,\"ry_cm\":5.3,\"j_cm4\":81.6,\"cw_cm6\":249963.42},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 58\",\"estado\":\"Revisado\",\"masa_kg_m\":86.6,\"d_ala_diametro_cm\":22.2,\"bf_b_cm\":20.9,\"tw_cm\":1.3,\"tf_t_cm\":2.06,\"area_cm2\":110.3,\"ix_cm4\":9490,\"iy_cm4\":3126,\"sx_cm3\":854.9549549549549,\"sy_cm3\":299.1387559808613,\"rx_cm\":9.3,\"ry_cm\":5.3,\"j_cm4\":139,\"cw_cm6\":316991.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 67\",\"estado\":\"Revisado\",\"masa_kg_m\":99.8,\"d_ala_diametro_cm\":22.9,\"bf_b_cm\":21,\"tw_cm\":1.45,\"tf_t_cm\":2.37,\"area_cm2\":127.1,\"ix_cm4\":11321,\"iy_cm4\":3688,\"sx_cm3\":988.7336244541485,\"sy_cm3\":351.23809523809524,\"rx_cm\":9.4,\"ry_cm\":5.4,\"j_cm4\":211,\"cw_cm6\":388605.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 12\",\"estado\":\"Con restricciones\",\"masa_kg_m\":17.9,\"d_ala_diametro_cm\":25.1,\"bf_b_cm\":10.1,\"tw_cm\":0.48,\"tf_t_cm\":0.53,\"area_cm2\":22.8,\"ix_cm4\":2239,\"iy_cm4\":91,\"sx_cm3\":178.40637450199202,\"sy_cm3\":18.01980198019802,\"rx_cm\":9.9,\"ry_cm\":2,\"j_cm4\":null,\"cw_cm6\":13733.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 15\",\"estado\":\"Con restricciones\",\"masa_kg_m\":22.3,\"d_ala_diametro_cm\":25.4,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.6900000000000001,\"area_cm2\":28.5,\"ix_cm4\":null,\"iy_cm4\":120,\"sx_cm3\":null,\"sy_cm3\":23.529411764705884,\"rx_cm\":null,\"ry_cm\":2.1,\"j_cm4\":4.2,\"cw_cm6\":18317.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 17\",\"estado\":\"Revisado\",\"masa_kg_m\":25.3,\"d_ala_diametro_cm\":25.7,\"bf_b_cm\":10.2,\"tw_cm\":0.61,\"tf_t_cm\":0.8400000000000001,\"area_cm2\":32.2,\"ix_cm4\":3409,\"iy_cm4\":148,\"sx_cm3\":265.2918287937743,\"sy_cm3\":29.01960784313726,\"rx_cm\":10.3,\"ry_cm\":2.1,\"j_cm4\":6.7,\"cw_cm6\":22866.73},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 19\",\"estado\":\"Revisado\",\"masa_kg_m\":28.5,\"d_ala_diametro_cm\":26,\"bf_b_cm\":10.2,\"tw_cm\":0.64,\"tf_t_cm\":1,\"area_cm2\":36.3,\"ix_cm4\":4008,\"iy_cm4\":176,\"sx_cm3\":308.3076923076923,\"sy_cm3\":34.50980392156863,\"rx_cm\":10.5,\"ry_cm\":2.2,\"j_cm4\":9.6,\"cw_cm6\":27500},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.9,\"d_ala_diametro_cm\":25.8,\"bf_b_cm\":14.6,\"tw_cm\":0.61,\"tf_t_cm\":0.9099999999999999,\"area_cm2\":41.9,\"ix_cm4\":4912,\"iy_cm4\":475,\"sx_cm3\":380.7751937984496,\"sy_cm3\":65.06849315068493,\"rx_cm\":10.8,\"ry_cm\":3.4,\"j_cm4\":10,\"cw_cm6\":73567.06},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.5,\"d_ala_diametro_cm\":26.2,\"bf_b_cm\":14.7,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":49.1,\"ix_cm4\":5994,\"iy_cm4\":587,\"sx_cm3\":457.55725190839695,\"sy_cm3\":79.8639455782313,\"rx_cm\":11,\"ry_cm\":3.5,\"j_cm4\":16.6,\"cw_cm6\":92306.69},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.8,\"d_ala_diametro_cm\":26.6,\"bf_b_cm\":14.8,\"tw_cm\":0.76,\"tf_t_cm\":1.3,\"area_cm2\":57,\"ix_cm4\":7076,\"iy_cm4\":695,\"sx_cm3\":532.0300751879699,\"sy_cm3\":93.91891891891892,\"rx_cm\":11.1,\"ry_cm\":3.5,\"j_cm4\":25.8,\"cw_cm6\":111215.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 33\",\"estado\":\"Revisado\",\"masa_kg_m\":49.2,\"d_ala_diametro_cm\":24.7,\"bf_b_cm\":20.2,\"tw_cm\":0.74,\"tf_t_cm\":1.1,\"area_cm2\":62.6,\"ix_cm4\":7076,\"iy_cm4\":1523,\"sx_cm3\":572.9554655870445,\"sy_cm3\":150.7920792079208,\"rx_cm\":10.6,\"ry_cm\":4.9,\"j_cm4\":24.1,\"cw_cm6\":212062.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 39\",\"estado\":\"Revisado\",\"masa_kg_m\":58.2,\"d_ala_diametro_cm\":25.2,\"bf_b_cm\":20.3,\"tw_cm\":0.8,\"tf_t_cm\":1.35,\"area_cm2\":74.2,\"ix_cm4\":8699,\"iy_cm4\":1873,\"sx_cm3\":690.3968253968254,\"sy_cm3\":184.53201970443348,\"rx_cm\":10.8,\"ry_cm\":5,\"j_cm4\":40.8,\"cw_cm6\":266351.14},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 45\",\"estado\":\"Revisado\",\"masa_kg_m\":67.4,\"d_ala_diametro_cm\":25.7,\"bf_b_cm\":20.2,\"tw_cm\":0.89,\"tf_t_cm\":1.6,\"area_cm2\":85.8,\"ix_cm4\":10323,\"iy_cm4\":2223,\"sx_cm3\":803.3463035019455,\"sy_cm3\":220.0990099009901,\"rx_cm\":11,\"ry_cm\":5.1,\"j_cm4\":62.9,\"cw_cm6\":322785.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 49\",\"estado\":\"Revisado\",\"masa_kg_m\":72.9,\"d_ala_diametro_cm\":25.3,\"bf_b_cm\":25.4,\"tw_cm\":0.86,\"tf_t_cm\":1.42,\"area_cm2\":92.9,\"ix_cm4\":11321,\"iy_cm4\":3888,\"sx_cm3\":894.9407114624506,\"sy_cm3\":306.1417322834646,\"rx_cm\":11,\"ry_cm\":6.5,\"j_cm4\":57.9,\"cw_cm6\":554287.28},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 54\",\"estado\":\"Revisado\",\"masa_kg_m\":80,\"d_ala_diametro_cm\":25.6,\"bf_b_cm\":25.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.56,\"area_cm2\":101.9,\"ix_cm4\":12612,\"iy_cm4\":4287,\"sx_cm3\":985.3125,\"sy_cm3\":336.2352941176471,\"rx_cm\":11.1,\"ry_cm\":6.5,\"j_cm4\":75.8,\"cw_cm6\":619387.47},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 60\",\"estado\":\"Revisado\",\"masa_kg_m\":89.1,\"d_ala_diametro_cm\":26,\"bf_b_cm\":25.6,\"tw_cm\":1.0699999999999998,\"tf_t_cm\":1.73,\"area_cm2\":113.6,\"ix_cm4\":14193,\"iy_cm4\":4828,\"sx_cm3\":1091.7692307692307,\"sy_cm3\":377.1875,\"rx_cm\":11.2,\"ry_cm\":6.5,\"j_cm4\":103,\"cw_cm6\":710962.71},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":26.4,\"bf_b_cm\":25.7,\"tw_cm\":1.19,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":129,\"ix_cm4\":16400,\"iy_cm4\":5577,\"sx_cm3\":1242.4242424242425,\"sy_cm3\":434.0077821011673,\"rx_cm\":11.3,\"ry_cm\":6.6,\"j_cm4\":148,\"cw_cm6\":832804.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 77\",\"estado\":\"Revisado\",\"masa_kg_m\":114.5,\"d_ala_diametro_cm\":26.9,\"bf_b_cm\":25.9,\"tw_cm\":1.35,\"tf_t_cm\":2.21,\"area_cm2\":145.8,\"ix_cm4\":18938,\"iy_cm4\":6410,\"sx_cm3\":1408.0297397769518,\"sy_cm3\":494.98069498069503,\"rx_cm\":11.4,\"ry_cm\":6.6,\"j_cm4\":213,\"cw_cm6\":976877.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 88\",\"estado\":\"Revisado\",\"masa_kg_m\":131.2,\"d_ala_diametro_cm\":27.5,\"bf_b_cm\":26.1,\"tw_cm\":1.54,\"tf_t_cm\":2.5100000000000002,\"area_cm2\":167.1,\"ix_cm4\":22227,\"iy_cm4\":7451,\"sx_cm3\":1616.509090909091,\"sy_cm3\":570.9578544061302,\"rx_cm\":11.5,\"ry_cm\":6.7,\"j_cm4\":313,\"cw_cm6\":1163287.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 100\",\"estado\":\"Revisado\",\"masa_kg_m\":148.9,\"d_ala_diametro_cm\":28.2,\"bf_b_cm\":26.3,\"tw_cm\":1.73,\"tf_t_cm\":2.84,\"area_cm2\":189.7,\"ix_cm4\":25931,\"iy_cm4\":8616,\"sx_cm3\":1839.0780141843973,\"sy_cm3\":655.2091254752852,\"rx_cm\":11.7,\"ry_cm\":6.7,\"j_cm4\":454,\"cw_cm6\":1385301.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 112\",\"estado\":\"Revisado\",\"masa_kg_m\":166.6,\"d_ala_diametro_cm\":28.9,\"bf_b_cm\":26.5,\"tw_cm\":1.92,\"tf_t_cm\":3.18,\"area_cm2\":212.3,\"ix_cm4\":29802,\"iy_cm4\":9823,\"sx_cm3\":2062.42214532872,\"sy_cm3\":741.3584905660377,\"rx_cm\":11.8,\"ry_cm\":6.8,\"j_cm4\":629,\"cw_cm6\":1624523.81},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 14\",\"estado\":\"Con restricciones\",\"masa_kg_m\":21.1,\"d_ala_diametro_cm\":30.3,\"bf_b_cm\":10.1,\"tw_cm\":0.5,\"tf_t_cm\":0.5700000000000001,\"area_cm2\":26.8,\"ix_cm4\":3688,\"iy_cm4\":98,\"sx_cm3\":243.43234323432344,\"sy_cm3\":19.405940594059405,\"rx_cm\":11.7,\"ry_cm\":1.9,\"j_cm4\":2.9,\"cw_cm6\":21654.89},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 16\",\"estado\":\"Revisado\",\"masa_kg_m\":23.9,\"d_ala_diametro_cm\":30.5,\"bf_b_cm\":10.1,\"tw_cm\":0.5599999999999999,\"tf_t_cm\":0.67,\"area_cm2\":30.4,\"ix_cm4\":4287,\"iy_cm4\":117,\"sx_cm3\":281.11475409836066,\"sy_cm3\":23.168316831683168,\"rx_cm\":11.9,\"ry_cm\":2,\"j_cm4\":4.2,\"cw_cm6\":26027.5},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 19\",\"estado\":\"Con restricciones\",\"masa_kg_m\":28.2,\"d_ala_diametro_cm\":30.9,\"bf_b_cm\":10.2,\"tw_cm\":0.6,\"tf_t_cm\":0.89,\"area_cm2\":35.9,\"ix_cm4\":5411,\"iy_cm4\":157,\"sx_cm3\":350.2265372168285,\"sy_cm3\":30.784313725490197,\"rx_cm\":12.2,\"ry_cm\":2.1,\"j_cm4\":7.5,\"cw_cm6\":35348.55},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.8,\"d_ala_diametro_cm\":31.3,\"bf_b_cm\":10.2,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.08,\"area_cm2\":41.8,\"ix_cm4\":6493,\"iy_cm4\":194,\"sx_cm3\":414.888178913738,\"sy_cm3\":38.03921568627451,\"rx_cm\":12.5,\"ry_cm\":2.2,\"j_cm4\":12,\"cw_cm6\":44292.55},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.7,\"d_ala_diametro_cm\":31,\"bf_b_cm\":16.5,\"tw_cm\":0.58,\"tf_t_cm\":0.97,\"area_cm2\":49.4,\"ix_cm4\":8491,\"iy_cm4\":720,\"sx_cm3\":547.8064516129032,\"sy_cm3\":87.27272727272727,\"rx_cm\":13.1,\"ry_cm\":3.8,\"j_cm4\":12.5,\"cw_cm6\":162324.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.5,\"d_ala_diametro_cm\":31.3,\"bf_b_cm\":16.6,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":56.7,\"ix_cm4\":9906,\"iy_cm4\":845,\"sx_cm3\":632.9712460063897,\"sy_cm3\":101.80722891566265,\"rx_cm\":13.2,\"ry_cm\":3.9,\"j_cm4\":19.1,\"cw_cm6\":192413.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":31.8,\"bf_b_cm\":16.7,\"tw_cm\":0.76,\"tf_t_cm\":1.3199999999999998,\"area_cm2\":66.5,\"ix_cm4\":11863,\"iy_cm4\":1020,\"sx_cm3\":746.1006289308176,\"sy_cm3\":122.1556886227545,\"rx_cm\":13.3,\"ry_cm\":3.9,\"j_cm4\":30.8,\"cw_cm6\":236902.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":30.3,\"bf_b_cm\":20.3,\"tw_cm\":0.75,\"tf_t_cm\":1.31,\"area_cm2\":76.1,\"ix_cm4\":12903,\"iy_cm4\":1836,\"sx_cm3\":851.6831683168317,\"sy_cm3\":180.88669950738915,\"rx_cm\":13,\"ry_cm\":4.9,\"j_cm4\":39.5,\"cw_cm6\":385752.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 45\",\"estado\":\"Revisado\",\"masa_kg_m\":66.9,\"d_ala_diametro_cm\":30.6,\"bf_b_cm\":20.4,\"tw_cm\":0.85,\"tf_t_cm\":1.46,\"area_cm2\":85.2,\"ix_cm4\":14568,\"iy_cm4\":2081,\"sx_cm3\":952.156862745098,\"sy_cm3\":204.01960784313727,\"rx_cm\":13.1,\"ry_cm\":4.9,\"j_cm4\":54.5,\"cw_cm6\":441764.88},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":31,\"bf_b_cm\":20.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.6300000000000001,\"area_cm2\":94.8,\"ix_cm4\":16400,\"iy_cm4\":2343,\"sx_cm3\":1058.0645161290322,\"sy_cm3\":228.58536585365854,\"rx_cm\":13.2,\"ry_cm\":5,\"j_cm4\":74.1,\"cw_cm6\":505266.13},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 53\",\"estado\":\"Revisado\",\"masa_kg_m\":79,\"d_ala_diametro_cm\":30.6,\"bf_b_cm\":25.4,\"tw_cm\":0.8800000000000001,\"tf_t_cm\":1.46,\"area_cm2\":100.7,\"ix_cm4\":17690,\"iy_cm4\":3987,\"sx_cm3\":1156.2091503267973,\"sy_cm3\":313.93700787401576,\"rx_cm\":13.3,\"ry_cm\":6.3,\"j_cm4\":65.8,\"cw_cm6\":846379.9},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 58\",\"estado\":\"Revisado\",\"masa_kg_m\":86.1,\"d_ala_diametro_cm\":31,\"bf_b_cm\":25.4,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.6300000000000001,\"area_cm2\":109.7,\"ix_cm4\":19771,\"iy_cm4\":4454,\"sx_cm3\":1275.5483870967741,\"sy_cm3\":350.70866141732284,\"rx_cm\":13.4,\"ry_cm\":6.4,\"j_cm4\":87.4,\"cw_cm6\":960501.65},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 65\",\"estado\":\"Revisado\",\"masa_kg_m\":96.7,\"d_ala_diametro_cm\":30.8,\"bf_b_cm\":30.5,\"tw_cm\":0.99,\"tf_t_cm\":1.54,\"area_cm2\":123.2,\"ix_cm4\":22185,\"iy_cm4\":7242,\"sx_cm3\":1440.5844155844156,\"sy_cm3\":474.88524590163934,\"rx_cm\":13.4,\"ry_cm\":7.7,\"j_cm4\":90.7,\"cw_cm6\":1550055.23},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 72\",\"estado\":\"Revisado\",\"masa_kg_m\":106.9,\"d_ala_diametro_cm\":31.1,\"bf_b_cm\":30.6,\"tw_cm\":1.09,\"tf_t_cm\":1.7,\"area_cm2\":136.1,\"ix_cm4\":24850,\"iy_cm4\":8116,\"sx_cm3\":1598.0707395498391,\"sy_cm3\":530.4575163398692,\"rx_cm\":13.5,\"ry_cm\":7.7,\"j_cm4\":122,\"cw_cm6\":1753786.44},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 79\",\"estado\":\"Revisado\",\"masa_kg_m\":117.5,\"d_ala_diametro_cm\":31.4,\"bf_b_cm\":30.7,\"tw_cm\":1.19,\"tf_t_cm\":1.8699999999999999,\"area_cm2\":149.7,\"ix_cm4\":27554,\"iy_cm4\":8991,\"sx_cm3\":1755.031847133758,\"sy_cm3\":585.7328990228013,\"rx_cm\":13.6,\"ry_cm\":7.7,\"j_cm4\":160,\"cw_cm6\":1960084.98},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 87\",\"estado\":\"Revisado\",\"masa_kg_m\":129.7,\"d_ala_diametro_cm\":31.8,\"bf_b_cm\":30.8,\"tw_cm\":1.31,\"tf_t_cm\":2.06,\"area_cm2\":165.2,\"ix_cm4\":30801,\"iy_cm4\":10031,\"sx_cm3\":1937.1698113207547,\"sy_cm3\":651.3636363636364,\"rx_cm\":13.7,\"ry_cm\":7.8,\"j_cm4\":212,\"cw_cm6\":2218023.62},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 96\",\"estado\":\"Revisado\",\"masa_kg_m\":142.8,\"d_ala_diametro_cm\":32.3,\"bf_b_cm\":30.9,\"tw_cm\":1.4,\"tf_t_cm\":2.29,\"area_cm2\":181.9,\"ix_cm4\":34672,\"iy_cm4\":11238,\"sx_cm3\":2146.8730650154803,\"sy_cm3\":727.3786407766991,\"rx_cm\":13.8,\"ry_cm\":7.8,\"j_cm4\":286,\"cw_cm6\":2530235.98},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 106\",\"estado\":\"Revisado\",\"masa_kg_m\":158,\"d_ala_diametro_cm\":32.7,\"bf_b_cm\":31,\"tw_cm\":1.55,\"tf_t_cm\":2.5100000000000002,\"area_cm2\":201.3,\"ix_cm4\":38834,\"iy_cm4\":12529,\"sx_cm3\":2375.1681957186543,\"sy_cm3\":808.3225806451613,\"rx_cm\":13.9,\"ry_cm\":7.9,\"j_cm4\":380,\"cw_cm6\":2854845.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 120\",\"estado\":\"Revisado\",\"masa_kg_m\":178.8,\"d_ala_diametro_cm\":33.3,\"bf_b_cm\":31.2,\"tw_cm\":1.8,\"tf_t_cm\":2.81,\"area_cm2\":227.8,\"ix_cm4\":44537,\"iy_cm4\":14360,\"sx_cm3\":2674.894894894895,\"sy_cm3\":920.5128205128206,\"rx_cm\":14,\"ry_cm\":8,\"j_cm4\":537,\"cw_cm6\":3337407.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.9,\"d_ala_diametro_cm\":34.9,\"bf_b_cm\":12.7,\"tw_cm\":0.58,\"tf_t_cm\":0.85,\"area_cm2\":41.9,\"ix_cm4\":8283,\"iy_cm4\":291,\"sx_cm3\":474.6704871060172,\"sy_cm3\":45.82677165354331,\"rx_cm\":14.1,\"ry_cm\":2.6,\"j_cm4\":8.7,\"cw_cm6\":84346.53},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.9,\"d_ala_diametro_cm\":35.3,\"bf_b_cm\":12.8,\"tw_cm\":0.64,\"tf_t_cm\":1.0699999999999998,\"area_cm2\":49.6,\"ix_cm4\":10198,\"iy_cm4\":371,\"sx_cm3\":577.7903682719548,\"sy_cm3\":57.96875,\"rx_cm\":14.4,\"ry_cm\":2.7,\"j_cm4\":15,\"cw_cm6\":108674.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.8,\"d_ala_diametro_cm\":35.2,\"bf_b_cm\":17.1,\"tw_cm\":0.6900000000000001,\"tf_t_cm\":0.9800000000000001,\"area_cm2\":57.1,\"ix_cm4\":12112,\"iy_cm4\":816,\"sx_cm3\":688.1818181818181,\"sy_cm3\":95.43859649122807,\"rx_cm\":14.6,\"ry_cm\":3.8,\"j_cm4\":15.8,\"cw_cm6\":238885.71},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 34\",\"estado\":\"Revisado\",\"masa_kg_m\":50.6,\"d_ala_diametro_cm\":35.5,\"bf_b_cm\":17.1,\"tw_cm\":0.72,\"tf_t_cm\":1.16,\"area_cm2\":64.5,\"ix_cm4\":14152,\"iy_cm4\":970,\"sx_cm3\":797.2957746478874,\"sy_cm3\":113.4502923976608,\"rx_cm\":14.8,\"ry_cm\":3.9,\"j_cm4\":23.7,\"cw_cm6\":285964.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 38\",\"estado\":\"Revisado\",\"masa_kg_m\":56.7,\"d_ala_diametro_cm\":35.8,\"bf_b_cm\":17.2,\"tw_cm\":0.79,\"tf_t_cm\":1.31,\"area_cm2\":72.3,\"ix_cm4\":16025,\"iy_cm4\":1111,\"sx_cm3\":895.2513966480448,\"sy_cm3\":129.18604651162792,\"rx_cm\":14.9,\"ry_cm\":3.9,\"j_cm4\":33.3,\"cw_cm6\":330400.32},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 43\",\"estado\":\"Revisado\",\"masa_kg_m\":63.8,\"d_ala_diametro_cm\":34.7,\"bf_b_cm\":20.3,\"tw_cm\":0.78,\"tf_t_cm\":1.35,\"area_cm2\":81.3,\"ix_cm4\":17815,\"iy_cm4\":1881,\"sx_cm3\":1026.801152737752,\"sy_cm3\":185.32019704433498,\"rx_cm\":14.8,\"ry_cm\":4.8,\"j_cm4\":43.7,\"cw_cm6\":523022.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 48\",\"estado\":\"Revisado\",\"masa_kg_m\":71.4,\"d_ala_diametro_cm\":35,\"bf_b_cm\":20.4,\"tw_cm\":0.86,\"tf_t_cm\":1.51,\"area_cm2\":91,\"ix_cm4\":20187,\"iy_cm4\":2139,\"sx_cm3\":1153.5428571428572,\"sy_cm3\":209.7058823529412,\"rx_cm\":14.9,\"ry_cm\":4.9,\"j_cm4\":60.8,\"cw_cm6\":599764.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 53\",\"estado\":\"Revisado\",\"masa_kg_m\":79,\"d_ala_diametro_cm\":35.4,\"bf_b_cm\":20.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.6800000000000002,\"area_cm2\":100.7,\"ix_cm4\":22518,\"iy_cm4\":2402,\"sx_cm3\":1272.2033898305085,\"sy_cm3\":234.34146341463415,\"rx_cm\":15,\"ry_cm\":4.9,\"j_cm4\":80.7,\"cw_cm6\":682791.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 61\",\"estado\":\"Recalculado\",\"masa_kg_m\":90.7,\"d_ala_diametro_cm\":35.3,\"bf_b_cm\":25.4,\"tw_cm\":0.95,\"tf_t_cm\":1.64,\"area_cm2\":115.5,\"ix_cm4\":26640,\"iy_cm4\":4454,\"sx_cm3\":1509.3484419263457,\"sy_cm3\":350.70866141732284,\"rx_cm\":15.2,\"ry_cm\":6.2,\"j_cm4\":91.6,\"cw_cm6\":1261590.6},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":35.7,\"bf_b_cm\":25.5,\"tw_cm\":1.05,\"tf_t_cm\":1.83,\"area_cm2\":129,\"ix_cm4\":30093,\"iy_cm4\":5036,\"sx_cm3\":1685.8823529411764,\"sy_cm3\":394.98039215686276,\"rx_cm\":15.3,\"ry_cm\":6.2,\"j_cm4\":125.7,\"cw_cm6\":1444295.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 74\",\"estado\":\"Revisado\",\"masa_kg_m\":110.4,\"d_ala_diametro_cm\":36,\"bf_b_cm\":25.6,\"tw_cm\":1.1400000000000001,\"tf_t_cm\":1.9899999999999998,\"area_cm2\":140.7,\"ix_cm4\":33132,\"iy_cm4\":5577,\"sx_cm3\":1840.6666666666667,\"sy_cm3\":435.703125,\"rx_cm\":15.3,\"ry_cm\":6.3,\"j_cm4\":161.5,\"cw_cm6\":1612701.23},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 82\",\"estado\":\"Revisado\",\"masa_kg_m\":122.1,\"d_ala_diametro_cm\":36.3,\"bf_b_cm\":25.7,\"tw_cm\":1.3,\"tf_t_cm\":2.17,\"area_cm2\":155.5,\"ix_cm4\":36711,\"iy_cm4\":6160,\"sx_cm3\":2022.6446280991738,\"sy_cm3\":479.3774319066148,\"rx_cm\":15.4,\"ry_cm\":6.3,\"j_cm4\":211.5,\"cw_cm6\":1793879.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.9,\"d_ala_diametro_cm\":39.9,\"bf_b_cm\":14,\"tw_cm\":0.64,\"tf_t_cm\":0.8800000000000001,\"area_cm2\":49.6,\"ix_cm4\":12529,\"iy_cm4\":399,\"sx_cm3\":628.0200501253133,\"sy_cm3\":57,\"rx_cm\":15.9,\"ry_cm\":2.8,\"j_cm4\":10.8,\"cw_cm6\":151875.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 31\",\"estado\":\"Recalculado\",\"masa_kg_m\":46.2,\"d_ala_diametro_cm\":40.3,\"bf_b_cm\":14,\"tw_cm\":0.7,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":58.8,\"ix_cm4\":15609,\"iy_cm4\":516,\"sx_cm3\":774.6401985111663,\"sy_cm3\":73.71428571428571,\"rx_cm\":16.3,\"ry_cm\":3,\"j_cm4\":19.1,\"cw_cm6\":198024.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 36\",\"estado\":\"Con restricciones\",\"masa_kg_m\":53.7,\"d_ala_diametro_cm\":40.3,\"bf_b_cm\":17.7,\"tw_cm\":0.75,\"tf_t_cm\":1.09,\"area_cm2\":68.4,\"ix_cm4\":18647,\"iy_cm4\":1020,\"sx_cm3\":925.409429280397,\"sy_cm3\":115.2542372881356,\"rx_cm\":16.5,\"ry_cm\":3.9,\"j_cm4\":22.5,\"cw_cm6\":392043.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 40\",\"estado\":\"Con restricciones\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":40.7,\"bf_b_cm\":17.8,\"tw_cm\":0.78,\"tf_t_cm\":1.28,\"area_cm2\":76.1,\"ix_cm4\":21561,\"iy_cm4\":1203,\"sx_cm3\":1059.5085995085994,\"sy_cm3\":135.1685393258427,\"rx_cm\":16.8,\"ry_cm\":4,\"j_cm4\":32.9,\"cw_cm6\":467346.37},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 45\",\"estado\":\"Con restricciones\",\"masa_kg_m\":67.4,\"d_ala_diametro_cm\":41,\"bf_b_cm\":17.9,\"tw_cm\":0.8800000000000001,\"tf_t_cm\":1.44,\"area_cm2\":85.8,\"ix_cm4\":24391,\"iy_cm4\":1365,\"sx_cm3\":1189.8048780487804,\"sy_cm3\":152.51396648044695,\"rx_cm\":16.9,\"ry_cm\":4,\"j_cm4\":46.2,\"cw_cm6\":534054.07},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 50\",\"estado\":\"Con restricciones\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":41.3,\"bf_b_cm\":18,\"tw_cm\":0.97,\"tf_t_cm\":1.6,\"area_cm2\":94.8,\"ix_cm4\":27430,\"iy_cm4\":1548,\"sx_cm3\":1328.3292978208233,\"sy_cm3\":172,\"rx_cm\":17,\"ry_cm\":4,\"j_cm4\":63.3,\"cw_cm6\":609946.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 57\",\"estado\":\"Con restricciones\",\"masa_kg_m\":83.1,\"d_ala_diametro_cm\":41.7,\"bf_b_cm\":18.1,\"tw_cm\":1.09,\"tf_t_cm\":1.8199999999999998,\"area_cm2\":108.4,\"ix_cm4\":31550,\"iy_cm4\":1794,\"sx_cm3\":1513.189448441247,\"sy_cm3\":198.232044198895,\"rx_cm\":17.1,\"ry_cm\":4.1,\"j_cm4\":92.4,\"cw_cm6\":713300.86},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 67\",\"estado\":\"Con restricciones\",\"masa_kg_m\":99.8,\"d_ala_diametro_cm\":41.5,\"bf_b_cm\":26,\"tw_cm\":1,\"tf_t_cm\":1.69,\"area_cm2\":127.1,\"ix_cm4\":39708,\"iy_cm4\":4953,\"sx_cm3\":1913.6385542168675,\"sy_cm3\":381,\"rx_cm\":17.7,\"ry_cm\":6.2,\"j_cm4\":99.5,\"cw_cm6\":1962423.3},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 77\",\"estado\":\"Con restricciones\",\"masa_kg_m\":114.5,\"d_ala_diametro_cm\":42,\"bf_b_cm\":26.1,\"tw_cm\":1.16,\"tf_t_cm\":1.9300000000000002,\"area_cm2\":145.8,\"ix_cm4\":46202,\"iy_cm4\":5744,\"sx_cm3\":2200.095238095238,\"sy_cm3\":440.15325670498083,\"rx_cm\":17.8,\"ry_cm\":6.3,\"j_cm4\":148.6,\"cw_cm6\":2305648.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 89\",\"estado\":\"Con restricciones\",\"masa_kg_m\":132.7,\"d_ala_diametro_cm\":42.5,\"bf_b_cm\":26.3,\"tw_cm\":1.3199999999999998,\"tf_t_cm\":2.2199999999999998,\"area_cm2\":169,\"ix_cm4\":54110,\"iy_cm4\":6785,\"sx_cm3\":2546.3529411764707,\"sy_cm3\":515.9695817490494,\"rx_cm\":17.9,\"ry_cm\":6.3,\"j_cm4\":226.9,\"cw_cm6\":2752128.99},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 100\",\"estado\":\"Con restricciones\",\"masa_kg_m\":148.9,\"d_ala_diametro_cm\":43.1,\"bf_b_cm\":26.5,\"tw_cm\":1.49,\"tf_t_cm\":2.5,\"area_cm2\":189.7,\"ix_cm4\":62018,\"iy_cm4\":7742,\"sx_cm3\":2877.8654292343385,\"sy_cm3\":584.3018867924528,\"rx_cm\":18,\"ry_cm\":6.4,\"j_cm4\":321.7,\"cw_cm6\":3190400.78},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":45,\"bf_b_cm\":15.2,\"tw_cm\":0.76,\"tf_t_cm\":1.08,\"area_cm2\":66.5,\"ix_cm4\":21228,\"iy_cm4\":637,\"sx_cm3\":943.4666666666667,\"sy_cm3\":83.81578947368422,\"rx_cm\":17.9,\"ry_cm\":3.1,\"j_cm4\":21.2,\"cw_cm6\":307187.9},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":45.3,\"bf_b_cm\":15.3,\"tw_cm\":0.8,\"tf_t_cm\":1.33,\"area_cm2\":76.1,\"ix_cm4\":25473,\"iy_cm4\":795,\"sx_cm3\":1124.635761589404,\"sy_cm3\":103.92156862745098,\"rx_cm\":18.3,\"ry_cm\":3.2,\"j_cm4\":33.7,\"cw_cm6\":384255.48},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 46\",\"estado\":\"Revisado\",\"masa_kg_m\":68.4,\"d_ala_diametro_cm\":45.9,\"bf_b_cm\":15.4,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.54,\"area_cm2\":87.1,\"ix_cm4\":29636,\"iy_cm4\":937,\"sx_cm3\":1291.3289760348584,\"sy_cm3\":121.68831168831169,\"rx_cm\":18.4,\"ry_cm\":3.3,\"j_cm4\":50.8,\"cw_cm6\":460959.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.3,\"d_ala_diametro_cm\":45.7,\"bf_b_cm\":19,\"tw_cm\":0.9,\"tf_t_cm\":1.45,\"area_cm2\":94.8,\"ix_cm4\":33298,\"iy_cm4\":1669,\"sx_cm3\":1457.2428884026258,\"sy_cm3\":175.68421052631578,\"rx_cm\":18.7,\"ry_cm\":4.2,\"j_cm4\":51.6,\"cw_cm6\":817001.58},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 55\",\"estado\":\"Revisado\",\"masa_kg_m\":82,\"d_ala_diametro_cm\":46,\"bf_b_cm\":19.1,\"tw_cm\":0.99,\"tf_t_cm\":1.6,\"area_cm2\":104.5,\"ix_cm4\":37045,\"iy_cm4\":1869,\"sx_cm3\":1610.6521739130435,\"sy_cm3\":195.7068062827225,\"rx_cm\":18.8,\"ry_cm\":4.2,\"j_cm4\":69.1,\"cw_cm6\":921117.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 60\",\"estado\":\"Revisado\",\"masa_kg_m\":89.1,\"d_ala_diametro_cm\":46.3,\"bf_b_cm\":19.2,\"tw_cm\":1.05,\"tf_t_cm\":1.77,\"area_cm2\":113.6,\"ix_cm4\":40957,\"iy_cm4\":2085,\"sx_cm3\":1769.2008639308856,\"sy_cm3\":217.1875,\"rx_cm\":18.9,\"ry_cm\":4.3,\"j_cm4\":90.3,\"cw_cm6\":1033597.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 65\",\"estado\":\"Revisado\",\"masa_kg_m\":96.7,\"d_ala_diametro_cm\":46.6,\"bf_b_cm\":19.3,\"tw_cm\":1.1400000000000001,\"tf_t_cm\":1.9100000000000001,\"area_cm2\":123.2,\"ix_cm4\":44537,\"iy_cm4\":2281,\"sx_cm3\":1911.4592274678112,\"sy_cm3\":236.37305699481865,\"rx_cm\":19,\"ry_cm\":4.3,\"j_cm4\":114,\"cw_cm6\":1138901.08},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 71\",\"estado\":\"Revisado\",\"masa_kg_m\":105.3,\"d_ala_diametro_cm\":46.9,\"bf_b_cm\":19.4,\"tw_cm\":1.26,\"tf_t_cm\":2.06,\"area_cm2\":134.2,\"ix_cm4\":48699,\"iy_cm4\":2510,\"sx_cm3\":2076.716417910448,\"sy_cm3\":258.7628865979382,\"rx_cm\":19.1,\"ry_cm\":4.3,\"j_cm4\":145,\"cw_cm6\":1261667.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 76\",\"estado\":\"Revisado\",\"masa_kg_m\":112.9,\"d_ala_diametro_cm\":46.3,\"bf_b_cm\":28,\"tw_cm\":1.08,\"tf_t_cm\":1.73,\"area_cm2\":143.9,\"ix_cm4\":55359,\"iy_cm4\":6327,\"sx_cm3\":2391.317494600432,\"sy_cm3\":451.92857142857144,\"rx_cm\":19.6,\"ry_cm\":6.6,\"j_cm4\":118,\"cw_cm6\":3142122.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 86\",\"estado\":\"Con restricciones\",\"masa_kg_m\":128.1,\"d_ala_diametro_cm\":46.7,\"bf_b_cm\":28.2,\"tw_cm\":1.22,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":163.2,\"ix_cm4\":63683,\"iy_cm4\":7284,\"sx_cm3\":2727.323340471092,\"sy_cm3\":516.5957446808511,\"rx_cm\":19.7,\"ry_cm\":6.7,\"j_cm4\":171,\"cw_cm6\":3645036.7},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 97\",\"estado\":\"Revisado\",\"masa_kg_m\":144.3,\"d_ala_diametro_cm\":47.2,\"bf_b_cm\":28.3,\"tw_cm\":1.3599999999999999,\"tf_t_cm\":2.21,\"area_cm2\":183.9,\"ix_cm4\":72840,\"iy_cm4\":8366,\"sx_cm3\":3086.4406779661017,\"sy_cm3\":591.2367491166077,\"rx_cm\":19.9,\"ry_cm\":6.7,\"j_cm4\":244,\"cw_cm6\":4233405.36},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 106\",\"estado\":\"Revisado\",\"masa_kg_m\":157.5,\"d_ala_diametro_cm\":47.6,\"bf_b_cm\":28.4,\"tw_cm\":1.5,\"tf_t_cm\":2.3899999999999997,\"area_cm2\":200.7,\"ix_cm4\":79500,\"iy_cm4\":9157,\"sx_cm3\":3340.3361344537816,\"sy_cm3\":644.8591549295775,\"rx_cm\":19.9,\"ry_cm\":6.8,\"j_cm4\":311,\"cw_cm6\":4679099.03},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 119\",\"estado\":\"Revisado\",\"masa_kg_m\":177.8,\"d_ala_diametro_cm\":48.2,\"bf_b_cm\":28.6,\"tw_cm\":1.6600000000000001,\"tf_t_cm\":2.69,\"area_cm2\":226.5,\"ix_cm4\":91154,\"iy_cm4\":10531,\"sx_cm3\":3782.323651452282,\"sy_cm3\":736.4335664335664,\"rx_cm\":20,\"ry_cm\":6.8,\"j_cm4\":441,\"cw_cm6\":5452846.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 44\",\"estado\":\"Revisado\",\"masa_kg_m\":65.8,\"d_ala_diametro_cm\":52.5,\"bf_b_cm\":16.5,\"tw_cm\":0.89,\"tf_t_cm\":1.1400000000000001,\"area_cm2\":83.9,\"ix_cm4\":35088,\"iy_cm4\":862,\"sx_cm3\":1336.6857142857143,\"sy_cm3\":104.48484848484848,\"rx_cm\":20.5,\"ry_cm\":3.2,\"j_cm4\":32,\"cw_cm6\":568456.59},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":52.9,\"bf_b_cm\":16.6,\"tw_cm\":0.97,\"tf_t_cm\":1.3599999999999999,\"area_cm2\":94.8,\"ix_cm4\":40957,\"iy_cm4\":1036,\"sx_cm3\":1548.468809073724,\"sy_cm3\":124.81927710843372,\"rx_cm\":20.8,\"ry_cm\":3.3,\"j_cm4\":47.5,\"cw_cm6\":688000.24},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 57\",\"estado\":\"Revisado\",\"masa_kg_m\":84.6,\"d_ala_diametro_cm\":53.5,\"bf_b_cm\":16.6,\"tw_cm\":1.03,\"tf_t_cm\":1.65,\"area_cm2\":107.7,\"ix_cm4\":48699,\"iy_cm4\":1274,\"sx_cm3\":1820.5233644859813,\"sy_cm3\":153.49397590361446,\"rx_cm\":21.2,\"ry_cm\":3.4,\"j_cm4\":73.7,\"cw_cm6\":856262.57},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 48\",\"estado\":\"Con restricciones\",\"masa_kg_m\":71.5,\"d_ala_diametro_cm\":52.4,\"bf_b_cm\":20.7,\"tw_cm\":0.89,\"tf_t_cm\":1.1,\"area_cm2\":91.8,\"ix_cm4\":39917,\"iy_cm4\":1611,\"sx_cm3\":1523.5496183206108,\"sy_cm3\":155.65217391304347,\"rx_cm\":20.9,\"ry_cm\":4.2,\"j_cm4\":33,\"cw_cm6\":1059913.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 55\",\"estado\":\"Con restricciones\",\"masa_kg_m\":81.9,\"d_ala_diametro_cm\":52.8,\"bf_b_cm\":20.9,\"tw_cm\":0.97,\"tf_t_cm\":1.33,\"area_cm2\":105,\"ix_cm4\":47451,\"iy_cm4\":2015,\"sx_cm3\":1797.3863636363637,\"sy_cm3\":192.82296650717706,\"rx_cm\":21.3,\"ry_cm\":4.4,\"j_cm4\":52,\"cw_cm6\":1334514.8},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 62\",\"estado\":\"Revisado\",\"masa_kg_m\":92.7,\"d_ala_diametro_cm\":53.3,\"bf_b_cm\":20.9,\"tw_cm\":1.02,\"tf_t_cm\":1.56,\"area_cm2\":118.1,\"ix_cm4\":55359,\"iy_cm4\":2393,\"sx_cm3\":2077.2607879924954,\"sy_cm3\":228.9952153110048,\"rx_cm\":21.7,\"ry_cm\":4.5,\"j_cm4\":76.2,\"cw_cm6\":1601531.76},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":53.7,\"bf_b_cm\":21,\"tw_cm\":1.09,\"tf_t_cm\":1.77,\"area_cm2\":129,\"ix_cm4\":61602,\"iy_cm4\":2693,\"sx_cm3\":2294.3016759776533,\"sy_cm3\":256.4761904761905,\"rx_cm\":21.8,\"ry_cm\":4.6,\"j_cm4\":102,\"cw_cm6\":1815570.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 73\",\"estado\":\"Revisado\",\"masa_kg_m\":108.9,\"d_ala_diametro_cm\":53.9,\"bf_b_cm\":21.1,\"tw_cm\":1.16,\"tf_t_cm\":1.8800000000000001,\"area_cm2\":138.7,\"ix_cm4\":66597,\"iy_cm4\":2939,\"sx_cm3\":2471.1317254174396,\"sy_cm3\":278.5781990521327,\"rx_cm\":21.9,\"ry_cm\":4.6,\"j_cm4\":126,\"cw_cm6\":1988292.57},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 83\",\"estado\":\"Revisado\",\"masa_kg_m\":123.1,\"d_ala_diametro_cm\":54.4,\"bf_b_cm\":21.2,\"tw_cm\":1.31,\"tf_t_cm\":2.12,\"area_cm2\":156.8,\"ix_cm4\":76170,\"iy_cm4\":3388,\"sx_cm3\":2800.3676470588234,\"sy_cm3\":319.62264150943395,\"rx_cm\":22,\"ry_cm\":4.6,\"j_cm4\":181,\"cw_cm6\":2315019.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 93\",\"estado\":\"Revisado\",\"masa_kg_m\":138.3,\"d_ala_diametro_cm\":54.9,\"bf_b_cm\":21.4,\"tw_cm\":1.47,\"tf_t_cm\":2.3600000000000003,\"area_cm2\":176.1,\"ix_cm4\":86160,\"iy_cm4\":3867,\"sx_cm3\":3138.7978142076504,\"sy_cm3\":361.4018691588785,\"rx_cm\":22.1,\"ry_cm\":4.7,\"j_cm4\":251,\"cw_cm6\":2668666.58},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 55\",\"estado\":\"Revisado\",\"masa_kg_m\":82,\"d_ala_diametro_cm\":59.9,\"bf_b_cm\":17.8,\"tw_cm\":1,\"tf_t_cm\":1.28,\"area_cm2\":104.5,\"ix_cm4\":56191,\"iy_cm4\":1211,\"sx_cm3\":1876.160267111853,\"sy_cm3\":136.06741573033707,\"rx_cm\":23.1,\"ry_cm\":3.4,\"j_cm4\":49,\"cw_cm6\":1040341.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 62\",\"estado\":\"Con restricciones\",\"masa_kg_m\":92.2,\"d_ala_diametro_cm\":60.3,\"bf_b_cm\":17.9,\"tw_cm\":1.09,\"tf_t_cm\":1.5,\"area_cm2\":117.4,\"ix_cm4\":64516,\"iy_cm4\":1436,\"sx_cm3\":2139.83416252073,\"sy_cm3\":160.44692737430168,\"rx_cm\":23.4,\"ry_cm\":3.5,\"j_cm4\":71,\"cw_cm6\":1241220.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.6,\"d_ala_diametro_cm\":60.3,\"bf_b_cm\":22.8,\"tw_cm\":1.05,\"tf_t_cm\":1.49,\"area_cm2\":129.7,\"ix_cm4\":76170,\"iy_cm4\":2930,\"sx_cm3\":2526.36815920398,\"sy_cm3\":257.0175438596491,\"rx_cm\":24.3,\"ry_cm\":4.7,\"j_cm4\":78,\"cw_cm6\":2533436.29},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 76\",\"estado\":\"Revisado\",\"masa_kg_m\":113.4,\"d_ala_diametro_cm\":60.8,\"bf_b_cm\":22.8,\"tw_cm\":1.1199999999999999,\"tf_t_cm\":1.73,\"area_cm2\":144.5,\"ix_cm4\":87408,\"iy_cm4\":3434,\"sx_cm3\":2875.263157894737,\"sy_cm3\":301.2280701754386,\"rx_cm\":24.6,\"ry_cm\":4.9,\"j_cm4\":112,\"cw_cm6\":2995533.92},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 84\",\"estado\":\"Revisado\",\"masa_kg_m\":125.1,\"d_ala_diametro_cm\":61.2,\"bf_b_cm\":22.9,\"tw_cm\":1.19,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":159.4,\"ix_cm4\":98647,\"iy_cm4\":3929,\"sx_cm3\":3223.7581699346406,\"sy_cm3\":343.14410480349346,\"rx_cm\":24.9,\"ry_cm\":5,\"j_cm4\":154,\"cw_cm6\":3447086.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 94\",\"estado\":\"Revisado\",\"masa_kg_m\":140.3,\"d_ala_diametro_cm\":61.7,\"bf_b_cm\":23,\"tw_cm\":1.31,\"tf_t_cm\":2.2199999999999998,\"area_cm2\":178.7,\"ix_cm4\":112382,\"iy_cm4\":4537,\"sx_cm3\":3642.8525121555913,\"sy_cm3\":394.5217391304348,\"rx_cm\":25.1,\"ry_cm\":5,\"j_cm4\":219,\"cw_cm6\":4012829.5},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 103\",\"estado\":\"Revisado\",\"masa_kg_m\":153.4,\"d_ala_diametro_cm\":62.3,\"bf_b_cm\":22.9,\"tw_cm\":1.4,\"tf_t_cm\":2.4899999999999998,\"area_cm2\":196,\"ix_cm4\":125000,\"iy_cm4\":5000,\"sx_cm3\":4012.841091492777,\"sy_cm3\":436.6812227074236,\"rx_cm\":25.3,\"ry_cm\":5.1,\"j_cm4\":295,\"cw_cm6\":4471545.13},{\"familia\":\"CE\",\"perfil_comercial\":\"CE\u1d2c 3 \u00d7 3.5\",\"estado\":\"Con restricciones\",\"masa_kg_m\":5.2,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":3.5799999999999996,\"tw_cm\":0.32,\"tf_t_cm\":0.645,\"area_cm2\":6.68,\"ix_cm4\":62.27,\"iy_cm4\":7.03,\"sx_cm3\":16.343832020997375,\"sy_cm3\":2.7677165354330713,\"rx_cm\":3.06,\"ry_cm\":1.03,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE\u1d2c 3 \u00d7 4.1\",\"estado\":\"Con restricciones\",\"masa_kg_m\":6.1,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":3.5799999999999996,\"tw_cm\":0.43,\"tf_t_cm\":0.693,\"area_cm2\":7.68,\"ix_cm4\":66.6,\"iy_cm4\":8.32,\"sx_cm3\":17.48031496062992,\"sy_cm3\":3.3643348160129403,\"rx_cm\":2.95,\"ry_cm\":1.04,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 4 \u00d7 4.5\",\"estado\":\"Con restricciones\",\"masa_kg_m\":6.7,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":4,\"tw_cm\":0.32,\"tf_t_cm\":0.752,\"area_cm2\":null,\"ix_cm4\":152,\"iy_cm4\":null,\"sx_cm3\":29.921259842519685,\"sy_cm3\":null,\"rx_cm\":4.14,\"ry_cm\":null,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 4 \u00d7 5.40\",\"estado\":\"Con restricciones\",\"masa_kg_m\":8.04,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":4.01,\"tw_cm\":0.45999999999999996,\"tf_t_cm\":0.752,\"area_cm2\":10.06,\"ix_cm4\":158.2,\"iy_cm4\":13.32,\"sx_cm3\":31.141732283464563,\"sy_cm3\":4.675324675324676,\"rx_cm\":3.98,\"ry_cm\":1.14,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 8.20\",\"estado\":\"Con restricciones\",\"masa_kg_m\":12.2,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":4.88,\"tw_cm\":0.51,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":15.4,\"ix_cm4\":535,\"iy_cm4\":27.6,\"sx_cm3\":70.20997375328083,\"sy_cm3\":7.645429362880887,\"rx_cm\":5.89,\"ry_cm\":1.34,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 10.50\",\"estado\":\"Con restricciones\",\"masa_kg_m\":15.63,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":5.1,\"tw_cm\":0.8,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":19.8,\"ix_cm4\":621,\"iy_cm4\":34.7,\"sx_cm3\":81.49606299212599,\"sy_cm3\":9.012987012987015,\"rx_cm\":5.6,\"ry_cm\":1.32,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 13.00\",\"estado\":\"Con restricciones\",\"masa_kg_m\":19.35,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":5.4,\"tw_cm\":1.1099999999999999,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":24.5,\"ix_cm4\":711,\"iy_cm4\":42,\"sx_cm3\":93.30708661417323,\"sy_cm3\":10.21897810218978,\"rx_cm\":5.39,\"ry_cm\":1.31,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3/4 \u00d7 1/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.88,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.11,\"ix_cm4\":0.37,\"iy_cm4\":0.37,\"sx_cm3\":0.27924528301886786,\"sy_cm3\":0.27924528301886786,\"rx_cm\":0.58,\"ry_cm\":0.58,\"j_cm4\":0.04,\"cw_cm6\":0.01},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.19,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.52,\"ix_cm4\":0.92,\"iy_cm4\":0.92,\"sx_cm3\":0.5168539325842697,\"sy_cm3\":0.5168539325842697,\"rx_cm\":0.79,\"ry_cm\":0.79,\"j_cm4\":0.05,\"cw_cm6\":0.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.73,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":2.21,\"ix_cm4\":1.25,\"iy_cm4\":1.25,\"sx_cm3\":0.7225433526011561,\"sy_cm3\":0.7225433526011561,\"rx_cm\":0.76,\"ry_cm\":0.76,\"j_cm4\":0.17,\"cw_cm6\":0.07},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 1/4\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.22,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":2.8,\"ix_cm4\":1.54,\"iy_cm4\":1.54,\"sx_cm3\":0.9166666666666666,\"sy_cm3\":0.9166666666666666,\"rx_cm\":0.74,\"ry_cm\":0.74,\"j_cm4\":0.41,\"cw_cm6\":0.16},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.5,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.93,\"ix_cm4\":1.83,\"iy_cm4\":1.83,\"sx_cm3\":0.800875273522976,\"sy_cm3\":0.800875273522976,\"rx_cm\":0.97,\"ry_cm\":0.97,\"j_cm4\":0.07,\"cw_cm6\":0.05},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.2,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":2.79,\"ix_cm4\":2.54,\"iy_cm4\":2.54,\"sx_cm3\":1.1519274376417232,\"sy_cm3\":1.1519274376417232,\"rx_cm\":0.97,\"ry_cm\":0.97,\"j_cm4\":0.22,\"cw_cm6\":0.15},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":2.86,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":3.72,\"ix_cm4\":3.21,\"iy_cm4\":3.21,\"sx_cm3\":1.4895591647331787,\"sy_cm3\":1.4895591647331787,\"rx_cm\":0.94,\"ry_cm\":0.94,\"j_cm4\":0.51,\"cw_cm6\":0.33},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 1/8\",\"estado\":\"Revisado\",\"masa_kg_m\":1.83,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":2.34,\"ix_cm4\":3.25,\"iy_cm4\":3.25,\"sx_cm3\":1.1861313868613137,\"sy_cm3\":1.1861313868613137,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.08,\"cw_cm6\":0.09},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 5/32\",\"estado\":\"Revisado\",\"masa_kg_m\":2.25,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.396875,\"area_cm2\":2.86,\"ix_cm4\":3.92,\"iy_cm4\":3.92,\"sx_cm3\":1.4464944649446494,\"sy_cm3\":1.4464944649446494,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.15,\"cw_cm6\":0.16},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.68,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":3.43,\"ix_cm4\":4.58,\"iy_cm4\":4.58,\"sx_cm3\":1.7026022304832715,\"sy_cm3\":1.7026022304832715,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.27,\"cw_cm6\":0.27},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":3.48,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":4.4,\"ix_cm4\":5.83,\"iy_cm4\":5.83,\"sx_cm3\":2.2251908396946565,\"sy_cm3\":2.2251908396946565,\"rx_cm\":1.14,\"ry_cm\":1.14,\"j_cm4\":0.62,\"cw_cm6\":0.61},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 3/4 \u00d7 1/8\",\"estado\":\"Revisado\",\"masa_kg_m\":2.14,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":4.445,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":2.74,\"ix_cm4\":5.41,\"iy_cm4\":5.41,\"sx_cm3\":1.677519379844961,\"sy_cm3\":1.677519379844961,\"rx_cm\":1.4,\"ry_cm\":1.4,\"j_cm4\":0.09,\"cw_cm6\":0.14},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 3/4 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":3.15,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":4.445,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":4.03,\"ix_cm4\":7.49,\"iy_cm4\":7.49,\"sx_cm3\":2.381558028616852,\"sy_cm3\":2.381558028616852,\"rx_cm\":1.37,\"ry_cm\":1.37,\"j_cm4\":0.31,\"cw_cm6\":0.45},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.46,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":3.1,\"ix_cm4\":7.91,\"iy_cm4\":7.91,\"sx_cm3\":2.1494565217391304,\"sy_cm3\":2.1494565217391304,\"rx_cm\":1.6,\"ry_cm\":1.6,\"j_cm4\":0.11,\"cw_cm6\":0.21},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":3.63,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":4.61,\"ix_cm4\":11.45,\"iy_cm4\":11.45,\"sx_cm3\":3.1542699724517904,\"sy_cm3\":3.1542699724517904,\"rx_cm\":1.57,\"ry_cm\":1.57,\"j_cm4\":0.36,\"cw_cm6\":0.68},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":6.06,\"ix_cm4\":14.57,\"iy_cm4\":14.57,\"sx_cm3\":4.069832402234637,\"sy_cm3\":4.069832402234637,\"rx_cm\":1.55,\"ry_cm\":1.55,\"j_cm4\":0.84,\"cw_cm6\":1.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":5.83,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":7.42,\"ix_cm4\":17.46,\"iy_cm4\":17.46,\"sx_cm3\":4.946175637393767,\"sy_cm3\":4.946175637393767,\"rx_cm\":1.52,\"ry_cm\":1.52,\"j_cm4\":1.63,\"cw_cm6\":2.85},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":6.99,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":8.77,\"ix_cm4\":19.98,\"iy_cm4\":19.98,\"sx_cm3\":5.791304347826086,\"sy_cm3\":5.791304347826086,\"rx_cm\":1.5,\"ry_cm\":1.5,\"j_cm4\":2.79,\"cw_cm6\":4.68},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 5/32\",\"estado\":\"Revisado\",\"masa_kg_m\":3.83,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.396875,\"area_cm2\":4.88,\"ix_cm4\":19.44,\"iy_cm4\":19.44,\"sx_cm3\":4.207792207792209,\"sy_cm3\":4.207792207792209,\"rx_cm\":1.98,\"ry_cm\":1.98,\"j_cm4\":0.26,\"cw_cm6\":0.81},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":4.61,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":5.81,\"ix_cm4\":22.89,\"iy_cm4\":22.89,\"sx_cm3\":4.97608695652174,\"sy_cm3\":4.97608695652174,\"rx_cm\":1.98,\"ry_cm\":1.98,\"j_cm4\":0.45,\"cw_cm6\":1.37},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":6.1,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":7.68,\"ix_cm4\":29.14,\"iy_cm4\":29.14,\"sx_cm3\":6.446902654867257,\"sy_cm3\":6.446902654867257,\"rx_cm\":1.96,\"ry_cm\":1.96,\"j_cm4\":1.06,\"cw_cm6\":3.12},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":7.44,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":9.48,\"ix_cm4\":35.38,\"iy_cm4\":35.38,\"sx_cm3\":7.914988814317675,\"sy_cm3\":7.914988814317675,\"rx_cm\":1.93,\"ry_cm\":1.93,\"j_cm4\":2.05,\"cw_cm6\":5.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":8.78,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":11.16,\"ix_cm4\":40.79,\"iy_cm4\":40.79,\"sx_cm3\":9.228506787330316,\"sy_cm3\":9.228506787330316,\"rx_cm\":1.91,\"ry_cm\":1.91,\"j_cm4\":3.52,\"cw_cm6\":9.73},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":5.52,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":7.03,\"ix_cm4\":40.01,\"iy_cm4\":40.01,\"sx_cm3\":7.222021660649819,\"sy_cm3\":7.222021660649819,\"rx_cm\":2.39,\"ry_cm\":2.39,\"j_cm4\":0.54,\"cw_cm6\":2.41},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":7.29,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":9.29,\"ix_cm4\":51.6,\"iy_cm4\":51.6,\"sx_cm3\":9.398907103825136,\"sy_cm3\":9.398907103825136,\"rx_cm\":2.36,\"ry_cm\":2.36,\"j_cm4\":1.27,\"cw_cm6\":5.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":9.08,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":11.48,\"ix_cm4\":62.9,\"iy_cm4\":62.9,\"sx_cm3\":11.626617375231053,\"sy_cm3\":11.626617375231053,\"rx_cm\":2.34,\"ry_cm\":2.34,\"j_cm4\":2.47,\"cw_cm6\":10.47},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":10.72,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":13.61,\"ix_cm4\":73.3,\"iy_cm4\":73.3,\"sx_cm3\":13.675373134328357,\"sy_cm3\":13.675373134328357,\"rx_cm\":2.31,\"ry_cm\":2.31,\"j_cm4\":4.25,\"cw_cm6\":17.5},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":13.99,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":17.74,\"ix_cm4\":92.4,\"iy_cm4\":92.4,\"sx_cm3\":17.566539923954373,\"sy_cm3\":17.566539923954373,\"rx_cm\":2.29,\"ry_cm\":2.29,\"j_cm4\":9.97,\"cw_cm6\":38.78},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":8.63,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":10.9,\"ix_cm4\":83.66,\"iy_cm4\":83.66,\"sx_cm3\":13.010886469673403,\"sy_cm3\":13.010886469673403,\"rx_cm\":2.77,\"ry_cm\":2.77,\"j_cm4\":1.49,\"cw_cm6\":8.96},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":10.71,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":13.48,\"ix_cm4\":101.95,\"iy_cm4\":101.95,\"sx_cm3\":15.979623824451409,\"sy_cm3\":15.979623824451409,\"rx_cm\":2.74,\"ry_cm\":2.74,\"j_cm4\":2.9,\"cw_cm6\":17.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":12.65,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":16,\"ix_cm4\":119.46,\"iy_cm4\":119.46,\"sx_cm3\":18.901898734177212,\"sy_cm3\":18.901898734177212,\"rx_cm\":2.72,\"ry_cm\":2.72,\"j_cm4\":4.98,\"cw_cm6\":28.6},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":16.52,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":20.97,\"ix_cm4\":151.51,\"iy_cm4\":151.51,\"sx_cm3\":24.43709677419354,\"sy_cm3\":24.43709677419354,\"rx_cm\":2.69,\"ry_cm\":2.69,\"j_cm4\":11.71,\"cw_cm6\":64.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":9.82,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":12.52,\"ix_cm4\":124.9,\"iy_cm4\":124.9,\"sx_cm3\":16.901217861975642,\"sy_cm3\":16.901217861975642,\"rx_cm\":3.18,\"ry_cm\":3.18,\"j_cm4\":1.71,\"cw_cm6\":13.56},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":12.2,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":15.48,\"ix_cm4\":154.4,\"iy_cm4\":154.4,\"sx_cm3\":21.092896174863387,\"sy_cm3\":21.092896174863387,\"rx_cm\":3.15,\"ry_cm\":3.15,\"j_cm4\":3.32,\"cw_cm6\":25.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":14.58,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":18.45,\"ix_cm4\":181.5,\"iy_cm4\":181.5,\"sx_cm3\":24.965612104539204,\"sy_cm3\":24.965612104539204,\"rx_cm\":3.12,\"ry_cm\":3.12,\"j_cm4\":5.72,\"cw_cm6\":43.6},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 7/16\",\"estado\":\"Revisado\",\"masa_kg_m\":16.82,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.11125,\"area_cm2\":21.35,\"ix_cm4\":206.9,\"iy_cm4\":206.9,\"sx_cm3\":28.656509695290858,\"sy_cm3\":28.656509695290858,\"rx_cm\":3.12,\"ry_cm\":3.12,\"j_cm4\":9.04,\"cw_cm6\":67.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":19.05,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":24.19,\"ix_cm4\":231.4,\"iy_cm4\":231.4,\"sx_cm3\":32.273361227336125,\"sy_cm3\":32.273361227336125,\"rx_cm\":3.1,\"ry_cm\":3.1,\"j_cm4\":13.44,\"cw_cm6\":98.34},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 5/8\",\"estado\":\"Revisado\",\"masa_kg_m\":23.36,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.5875,\"area_cm2\":29.74,\"ix_cm4\":277.2,\"iy_cm4\":277.2,\"sx_cm3\":39.375,\"sy_cm3\":39.375,\"rx_cm\":3.05,\"ry_cm\":3.05,\"j_cm4\":26.04,\"cw_cm6\":182.63},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 5/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":15.33,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":19.5,\"ix_cm4\":310.7,\"iy_cm4\":310.7,\"sx_cm3\":33.51672060409925,\"sy_cm3\":33.51672060409925,\"rx_cm\":3.96,\"ry_cm\":3.96,\"j_cm4\":4.27,\"cw_cm6\":52.93},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":18.3,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":23.29,\"ix_cm4\":363.8,\"iy_cm4\":363.8,\"sx_cm3\":39.67284623773173,\"sy_cm3\":39.67284623773173,\"rx_cm\":3.94,\"ry_cm\":3.94,\"j_cm4\":7.18,\"cw_cm6\":87.69},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":24.11,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":30.65,\"ix_cm4\":468.3,\"iy_cm4\":468.3,\"sx_cm3\":51.63175303197354,\"sy_cm3\":51.63175303197354,\"rx_cm\":3.91,\"ry_cm\":3.91,\"j_cm4\":16.91,\"cw_cm6\":199.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 0.109\",\"estado\":\"Revisado\",\"masa_kg_m\":1.04,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.27686,\"area_cm2\":1.33,\"ix_cm4\":0.81,\"iy_cm4\":0.81,\"sx_cm3\":0.45,\"sy_cm3\":0.45,\"rx_cm\":0.78,\"ry_cm\":0.78,\"j_cm4\":0.035,\"cw_cm6\":0.016},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 0.109\",\"estado\":\"Revisado\",\"masa_kg_m\":1.32,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.27686,\"area_cm2\":1.68,\"ix_cm4\":1.62,\"iy_cm4\":1.62,\"sx_cm3\":0.7120879120879121,\"sy_cm3\":0.7120879120879121,\"rx_cm\":0.98,\"ry_cm\":0.98,\"j_cm4\":0.044,\"cw_cm6\":0.033},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 0.109\",\"estado\":\"Pendiente de datos\",\"masa_kg_m\":null,\"d_ala_diametro_cm\":null,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":null,\"ix_cm4\":null,\"iy_cm4\":null,\"sx_cm3\":null,\"sy_cm3\":null,\"rx_cm\":null,\"ry_cm\":null,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 0.137\",\"estado\":\"Revisado\",\"masa_kg_m\":1.99,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.34798,\"area_cm2\":2.53,\"ix_cm4\":3.51,\"iy_cm4\":3.51,\"sx_cm3\":1.2857142857142856,\"sy_cm3\":1.2857142857142856,\"rx_cm\":1.18,\"ry_cm\":1.18,\"j_cm4\":0.105,\"cw_cm6\":0.113},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 0.232\",\"estado\":\"Revisado\",\"masa_kg_m\":4.43,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.58928,\"area_cm2\":5.64,\"ix_cm4\":13.6,\"iy_cm4\":13.6,\"sx_cm3\":3.788300835654596,\"sy_cm3\":3.788300835654596,\"rx_cm\":1.55,\"ry_cm\":1.55,\"j_cm4\":0.673,\"cw_cm6\":1.246},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 0.281\",\"estado\":\"Revisado\",\"masa_kg_m\":8.14,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.71374,\"area_cm2\":10.37,\"ix_cm4\":57.36,\"iy_cm4\":57.36,\"sx_cm3\":10.524770642201835,\"sy_cm3\":10.524770642201835,\"rx_cm\":2.35,\"ry_cm\":2.35,\"j_cm4\":1.804,\"cw_cm6\":7.74},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.95,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.209675,\"ix_cm4\":0.6503616025000001,\"iy_cm4\":0.02286427508789063,\"sx_cm3\":0.51209575,\"sy_cm3\":0.096017953125,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.08065484308535399,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.27,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6129,\"ix_cm4\":0.8671488033333334,\"iy_cm4\":0.05419680020833334,\"sx_cm3\":0.6827943333333333,\"sy_cm3\":0.17069858333333332,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.18265433128025108,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.58,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.0161249999999997,\"ix_cm4\":1.0839360041666668,\"iy_cm4\":0.10585312540690105,\"sx_cm3\":0.8534929166666666,\"sy_cm3\":0.2667165364583333,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.34011941341802115,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":1.3007232050000002,\"iy_cm4\":0.18291420070312503,\"sx_cm3\":1.0241915,\"sy_cm3\":0.3840718125,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.5590877376299159,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.53,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.2258,\"ix_cm4\":1.7342976066666669,\"iy_cm4\":0.4335744016666667,\"sx_cm3\":1.3655886666666666,\"sy_cm3\":0.6827943333333333,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.3666174209354124,\"j_cm4\":1.1908391925776043,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.19,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.51209375,\"ix_cm4\":1.2702375048828125,\"iy_cm4\":0.028580343859863282,\"sx_cm3\":0.800149609375,\"sy_cm3\":0.12002244140625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.1035184612275958,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.58,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.0161249999999997,\"ix_cm4\":1.6936500065104168,\"iy_cm4\":0.06774600026041667,\"sx_cm3\":1.0668661458333333,\"sy_cm3\":0.21337322916666665,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.23684456944163418,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.97,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.52015625,\"ix_cm4\":2.1170625081380208,\"iy_cm4\":0.1323164067586263,\"sx_cm3\":1.3335826822916665,\"sy_cm3\":0.33339567057291664,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.44593342597717534,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.37,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.0241875,\"ix_cm4\":2.540475009765625,\"iy_cm4\":0.22864275087890626,\"sx_cm3\":1.60029921875,\"sy_cm3\":0.480089765625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.7418337602469454,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.16,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.0322499999999994,\"ix_cm4\":3.3873000130208335,\"iy_cm4\":0.5419680020833334,\"sx_cm3\":2.1337322916666666,\"sy_cm3\":0.8534929166666666,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.3666174209354124,\"j_cm4\":1.6227337102250132,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.96,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.0403125,\"ix_cm4\":4.2341250162760415,\"iy_cm4\":1.0585312540690104,\"sx_cm3\":2.667165364583333,\"sy_cm3\":1.3335826822916665,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.45827177616926545,\"j_cm4\":2.907322247503916,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":5.08095001953125,\"iy_cm4\":1.82914200703125,\"sx_cm3\":3.2005984375,\"sy_cm3\":1.9203590625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.5499261314031186,\"j_cm4\":4.580774470811767,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.42,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.8145125,\"ix_cm4\":2.1949704084375,\"iy_cm4\":0.034296412631835936,\"sx_cm3\":1.1522154375,\"sy_cm3\":0.1440269296875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.1263825003434403,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":2.9266272112500005,\"iy_cm4\":0.08129520031250001,\"sx_cm3\":1.53628725,\"sy_cm3\":0.256047875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.2910390125909807,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.37,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.0241875,\"ix_cm4\":3.6582840140625,\"iy_cm4\":0.15877968811035156,\"sx_cm3\":1.9203590625,\"sy_cm3\":0.4000748046875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.5517725022177139,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.85,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.629025,\"ix_cm4\":4.389940816875,\"iy_cm4\":0.2743713010546875,\"sx_cm3\":2.304430875,\"sy_cm3\":0.57610771875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.924687552106271,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":5.853254422500001,\"iy_cm4\":0.6503616025000001,\"sx_cm3\":3.0725745,\"sy_cm3\":1.0241915,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.3666174209354124,\"j_cm4\":2.0557047047910495,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":7.316568028125,\"iy_cm4\":1.2702375048828125,\"sx_cm3\":3.840718125,\"sy_cm3\":1.60029921875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.45827177616926545,\"j_cm4\":3.750550664170321,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.7,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.25805,\"ix_cm4\":8.77988163375,\"iy_cm4\":2.1949704084375,\"sx_cm3\":4.60886175,\"sy_cm3\":2.304430875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.5499261314031186,\"j_cm4\":6.028623412424121,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":11.706508845000002,\"iy_cm4\":5.202892820000001,\"sx_cm3\":6.145149,\"sy_cm3\":4.096766,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.7332348418708248,\"j_cm4\":12.214593810508642,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":5.202892820000001,\"iy_cm4\":0.04572855017578126,\"sx_cm3\":2.048383,\"sy_cm3\":0.19203590625,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.17211090026864795,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.53,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.2258,\"ix_cm4\":6.9371904266666675,\"iy_cm4\":0.10839360041666668,\"sx_cm3\":2.731177333333333,\"sy_cm3\":0.34139716666666664,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.3994311121965522,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.16,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.0322499999999994,\"ix_cm4\":8.671488033333334,\"iy_cm4\":0.2117062508138021,\"sx_cm3\":3.4139716666666664,\"sy_cm3\":0.5334330729166666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.7634698075002958,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":10.405785640000001,\"iy_cm4\":0.36582840140625006,\"sx_cm3\":4.096766,\"sy_cm3\":0.768143625,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.2749630657015593,\"j_cm4\":1.2904774893656639,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.06,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":13.874380853333335,\"iy_cm4\":0.8671488033333334,\"sx_cm3\":5.462354666666666,\"sy_cm3\":1.3655886666666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.3666174209354124,\"j_cm4\":2.922469300484017,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":17.34297606666667,\"iy_cm4\":1.6936500065104168,\"sx_cm3\":6.827943333333333,\"sy_cm3\":2.1337322916666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.45827177616926545,\"j_cm4\":5.4419106146883385,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":20.811571280000003,\"iy_cm4\":2.9266272112500005,\"sx_cm3\":8.193532,\"sy_cm3\":3.0725745,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.5499261314031186,\"j_cm4\":8.945403802078655,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.13,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.9032,\"ix_cm4\":27.74876170666667,\"iy_cm4\":6.9371904266666675,\"sx_cm3\":10.924709333333332,\"sy_cm3\":5.462354666666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.7332348418708248,\"j_cm4\":19.05342708124167,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.95,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.0403125,\"ix_cm4\":16.936500065104166,\"iy_cm4\":0.2646328135172526,\"sx_cm3\":5.334330729166666,\"sy_cm3\":0.6667913411458333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.9751736137611134,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":20.323800078125,\"iy_cm4\":0.4572855017578125,\"sx_cm3\":6.401196875,\"sy_cm3\":0.96017953125,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.2749630657015593,\"j_cm4\":1.6562953796415327,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":27.098400104166668,\"iy_cm4\":1.0839360041666668,\"sx_cm3\":8.534929166666666,\"sy_cm3\":1.7069858333333332,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.3666174209354124,\"j_cm4\":3.789513111066147,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.91,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":33.87300013020833,\"iy_cm4\":2.1170625081380208,\"sx_cm3\":10.668661458333332,\"sy_cm3\":2.667165364583333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.45827177616926545,\"j_cm4\":7.134934815634805,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":40.64760015625,\"iy_cm4\":3.6582840140625,\"sx_cm3\":12.80239375,\"sy_cm3\":3.840718125,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.5499261314031186,\"j_cm4\":11.869340163951126,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":54.196800208333336,\"iy_cm4\":8.671488033333334,\"sx_cm3\":17.069858333333332,\"sy_cm3\":6.827943333333333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.7332348418708248,\"j_cm4\":25.96373936360021,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.82,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.16125,\"ix_cm4\":67.74600026041666,\"iy_cm4\":16.936500065104166,\"sx_cm3\":21.337322916666665,\"sy_cm3\":10.668661458333332,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.9165435523385309,\"j_cm4\":46.51715596006266,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":23.413017690000004,\"iy_cm4\":0.16259040062500002,\"sx_cm3\":6.145149,\"sy_cm3\":0.51209575,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.6162177555857645,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.74,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":29.2662721125,\"iy_cm4\":0.3175593762207031,\"sx_cm3\":7.68143625,\"sy_cm3\":0.800149609375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.22913588808463273,\"j_cm4\":1.186878986502018,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.7,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.25805,\"ix_cm4\":35.119526535,\"iy_cm4\":0.548742602109375,\"sx_cm3\":9.2177235,\"sy_cm3\":1.1522154375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.2749630657015593,\"j_cm4\":2.022120005495045,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":46.82603538000001,\"iy_cm4\":1.3007232050000002,\"sx_cm3\":12.290298,\"sy_cm3\":2.048383,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.3666174209354124,\"j_cm4\":4.656624201455691,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":58.532544225,\"iy_cm4\":2.540475009765625,\"sx_cm3\":15.3628725,\"sy_cm3\":3.2005984375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.45827177616926545,\"j_cm4\":8.828360035483422,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.39,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":70.23905307,\"iy_cm4\":4.389940816875,\"sx_cm3\":18.435447,\"sy_cm3\":4.60886175,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.5499261314031186,\"j_cm4\":14.795000833700335,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":93.65207076000002,\"iy_cm4\":10.405785640000001,\"sx_cm3\":24.580596,\"sy_cm3\":8.193532,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.7332348418708248,\"j_cm4\":32.89127527665679,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.06,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":55.49752341333334,\"iy_cm4\":0.21678720083333336,\"sx_cm3\":10.924709333333332,\"sy_cm3\":0.6827943333333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.8330048626184045,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.32,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":69.37190426666668,\"iy_cm4\":0.4234125016276042,\"sx_cm3\":13.655886666666666,\"sy_cm3\":1.0668661458333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.22913588808463273,\"j_cm4\":1.610290929033921,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":83.24628512000001,\"iy_cm4\":0.7316568028125001,\"sx_cm3\":16.387064,\"sy_cm3\":1.53628725,\"rx_cm\":2.932939367483299,\"ry_cm\":0.2749630657015593,\"j_cm4\":2.753774404298367,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.13,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.9032,\"ix_cm4\":110.99504682666668,\"iy_cm4\":1.7342976066666669,\"sx_cm3\":21.849418666666665,\"sy_cm3\":2.731177333333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.3666174209354124,\"j_cm4\":6.390897795144835,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":138.74380853333335,\"iy_cm4\":3.3873000130208335,\"sx_cm3\":27.31177333333333,\"sy_cm3\":4.267464583333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.45827177616926545,\"j_cm4\":12.215516920004733,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":166.49257024000002,\"iy_cm4\":5.853254422500001,\"sx_cm3\":32.774128,\"sy_cm3\":6.145149,\"rx_cm\":2.932939367483299,\"ry_cm\":0.5499261314031186,\"j_cm4\":20.647639829850622,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":20.26,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":25.8064,\"ix_cm4\":221.99009365333336,\"iy_cm4\":13.874380853333335,\"sx_cm3\":43.69883733333333,\"sy_cm3\":10.924709333333332,\"rx_cm\":2.932939367483299,\"ry_cm\":0.7332348418708248,\"j_cm4\":46.759508807744275,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":108.39360041666667,\"iy_cm4\":0.2709840010416667,\"sx_cm3\":17.069858333333332,\"sy_cm3\":0.8534929166666666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.1833087104677062,\"j_cm4\":1.0497920378187418,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.9,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":135.49200052083333,\"iy_cm4\":0.5292656270345052,\"sx_cm3\":21.337322916666665,\"sy_cm3\":1.3335826822916665,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.22913588808463273,\"j_cm4\":2.0337032778769633,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":162.590400625,\"iy_cm4\":0.914571003515625,\"sx_cm3\":25.6047875,\"sy_cm3\":1.9203590625,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.2749630657015593,\"j_cm4\":3.4854305501652174,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":216.78720083333334,\"iy_cm4\":2.1678720083333336,\"sx_cm3\":34.139716666666665,\"sy_cm3\":3.4139716666666664,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.3666174209354124,\"j_cm4\":8.12518883976455,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.82,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.16125,\"ix_cm4\":270.98400104166666,\"iy_cm4\":4.2341250162760415,\"sx_cm3\":42.67464583333333,\"sy_cm3\":5.334330729166666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.45827177616926545,\"j_cm4\":15.602777820177815,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":18.99,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":24.1935,\"ix_cm4\":325.18080125,\"iy_cm4\":7.316568028125,\"sx_cm3\":51.209575,\"sy_cm3\":7.68143625,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.5499261314031186,\"j_cm4\":26.500726074264524,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":25.32,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":32.257999999999996,\"ix_cm4\":433.5744016666667,\"iy_cm4\":17.34297606666667,\"sx_cm3\":68.27943333333333,\"sy_cm3\":13.655886666666666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.7332348418708248,\"j_cm4\":60.63220977705835,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":187.30414152000003,\"iy_cm4\":0.32518080125000004,\"sx_cm3\":24.580596,\"sy_cm3\":1.0241915,\"rx_cm\":4.399409051224949,\"ry_cm\":0.1833087104677062,\"j_cm4\":1.2665792294448135,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.48,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":234.1301769,\"iy_cm4\":0.6351187524414063,\"sx_cm3\":30.725745,\"sy_cm3\":1.60029921875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.22913588808463273,\"j_cm4\":2.4571157246250115,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.39,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":280.95621228,\"iy_cm4\":1.09748520421875,\"sx_cm3\":36.870894,\"sy_cm3\":2.304430875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.2749630657015593,\"j_cm4\":4.217087117005672,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":374.60828304000006,\"iy_cm4\":2.6014464100000003,\"sx_cm3\":49.161192,\"sy_cm3\":4.096766,\"rx_cm\":4.399409051224949,\"ry_cm\":0.3666174209354124,\"j_cm4\":9.859484089372232,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":18.99,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":24.1935,\"ix_cm4\":468.2603538,\"iy_cm4\":5.08095001953125,\"sx_cm3\":61.45149,\"sy_cm3\":6.401196875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.45827177616926545,\"j_cm4\":18.990063784032287,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":22.79,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":29.0322,\"ix_cm4\":561.91242456,\"iy_cm4\":8.77988163375,\"sx_cm3\":73.741788,\"sy_cm3\":9.2177235,\"rx_cm\":4.399409051224949,\"ry_cm\":0.5499261314031186,\"j_cm4\":32.35392008792072,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":30.38,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":38.7096,\"ix_cm4\":749.2165660800001,\"iy_cm4\":20.811571280000003,\"sx_cm3\":98.322384,\"sy_cm3\":16.387064,\"rx_cm\":4.399409051224949,\"ry_cm\":0.7332348418708248,\"j_cm4\":74.50598722329106,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.994,\"d_ala_diametro_cm\":1.27,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.2667686977437442,\"ix_cm4\":0.12769820203693033,\"iy_cm4\":0.12769820203693033,\"sx_cm3\":0.20109953076681944,\"sy_cm3\":0.20109953076681944,\"rx_cm\":0.3175,\"ry_cm\":0.3175,\"j_cm4\":0.25539640407386066,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 9/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.257,\"d_ala_diametro_cm\":1.42875,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6032541330819265,\"ix_cm4\":0.2045478280186279,\"iy_cm4\":0.2045478280186279,\"sx_cm3\":0.28633116783010026,\"sy_cm3\":0.28633116783010026,\"rx_cm\":0.3571875,\"ry_cm\":0.3571875,\"j_cm4\":0.4090956560372558,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.552,\"d_ala_diametro_cm\":1.5875,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.9793260902246004,\"ix_cm4\":0.31176318856672436,\"iy_cm4\":0.31176318856672436,\"sx_cm3\":0.3927725210289441,\"sy_cm3\":0.3927725210289441,\"rx_cm\":0.396875,\"ry_cm\":0.396875,\"j_cm4\":0.6235263771334487,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.235,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.8502295699234246,\"ix_cm4\":0.6464721478119598,\"iy_cm4\":0.6464721478119598,\"sx_cm3\":0.6787109163380155,\"sy_cm3\":0.6787109163380155,\"rx_cm\":0.47625,\"ry_cm\":0.47625,\"j_cm4\":1.2929442956239197,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 7/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.045,\"d_ala_diametro_cm\":2.2225,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.8794791368402177,\"ix_cm4\":1.1976694651979287,\"iy_cm4\":1.1976694651979287,\"sx_cm3\":1.077767797703423,\"sy_cm3\":1.077767797703423,\"rx_cm\":0.555625,\"ry_cm\":0.555625,\"j_cm4\":2.3953389303958574,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.973,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.067074790974977,\"ix_cm4\":2.0431712325908853,\"iy_cm4\":2.0431712325908853,\"sx_cm3\":1.6087962461345555,\"sy_cm3\":1.6087962461345555,\"rx_cm\":0.635,\"ry_cm\":0.635,\"j_cm4\":4.086342465181771,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.022,\"d_ala_diametro_cm\":2.8575,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.413016532327706,\"ix_cm4\":3.2727652482980463,\"iy_cm4\":3.2727652482980463,\"sx_cm3\":2.290649342640802,\"sy_cm3\":2.290649342640802,\"rx_cm\":0.714375,\"ry_cm\":0.714375,\"j_cm4\":6.5455304965960925,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.208,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.9173043608984015,\"ix_cm4\":4.98821101706759,\"iy_cm4\":4.98821101706759,\"sx_cm3\":3.142180168231553,\"sy_cm3\":3.142180168231553,\"rx_cm\":0.79375,\"ry_cm\":0.79375,\"j_cm4\":9.97642203413518,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":9,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":11.400918279693698,\"ix_cm4\":10.343554364991357,\"iy_cm4\":10.343554364991357,\"sx_cm3\":5.429687330704124,\"sy_cm3\":5.429687330704124,\"rx_cm\":0.9525,\"ry_cm\":0.9525,\"j_cm4\":20.687108729982715,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.49,\"d_ala_diametro_cm\":4.1275,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":13.380244369918303,\"ix_cm4\":14.246829485846753,\"iy_cm4\":14.246829485846753,\"sx_cm3\":6.903369829604725,\"sy_cm3\":6.903369829604725,\"rx_cm\":1.031875,\"ry_cm\":1.031875,\"j_cm4\":28.493658971693506,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.17,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":15.517916547360871,\"ix_cm4\":19.16271144316686,\"iy_cm4\":19.16271144316686,\"sx_cm3\":8.622142381627384,\"sy_cm3\":8.622142381627384,\"rx_cm\":1.11125,\"ry_cm\":1.11125,\"j_cm4\":38.32542288633372,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 2\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.89,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.268299163899908,\"ix_cm4\":32.690739721454165,\"iy_cm4\":32.690739721454165,\"sx_cm3\":12.870369969076444,\"sy_cm3\":12.870369969076444,\"rx_cm\":1.27,\"ry_cm\":1.27,\"j_cm4\":65.38147944290833,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.266,\"d_ala_diametro_cm\":1.27,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6129,\"ix_cm4\":0.21678720083333336,\"iy_cm4\":0.21678720083333336,\"sx_cm3\":0.3413971666666667,\"sy_cm3\":0.3413971666666667,\"rx_cm\":0.3666174209354124,\"ry_cm\":0.3666174209354124,\"j_cm4\":0.3663703694083334,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.978,\"d_ala_diametro_cm\":1.5875,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.52015625,\"ix_cm4\":0.5292656270345051,\"iy_cm4\":0.5292656270345051,\"sx_cm3\":0.6667913411458333,\"sy_cm3\":0.6667913411458333,\"rx_cm\":0.45827177616926545,\"ry_cm\":0.45827177616926545,\"j_cm4\":0.8944589096883138,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.848,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.629025,\"ix_cm4\":1.09748520421875,\"iy_cm4\":1.09748520421875,\"sx_cm3\":1.1522154375,\"sy_cm3\":1.1522154375,\"rx_cm\":0.5499261314031186,\"ry_cm\":0.5499261314031186,\"j_cm4\":1.8547499951296875,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.063,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":3.4685952133333338,\"iy_cm4\":3.4685952133333338,\"sx_cm3\":2.7311773333333336,\"sy_cm3\":2.7311773333333336,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.7332348418708248,\"j_cm4\":5.861925910533334,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.91,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":8.468250032552081,\"iy_cm4\":8.468250032552081,\"sx_cm3\":5.334330729166666,\"sy_cm3\":5.334330729166666,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.9165435523385309,\"j_cm4\":14.31134255501302,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.391,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":17.5597632675,\"iy_cm4\":17.5597632675,\"sx_cm3\":9.2177235,\"sy_cm3\":9.2177235,\"rx_cm\":1.0998522628062373,\"ry_cm\":1.0998522628062373,\"j_cm4\":29.675999922075,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 2\",\"estado\":\"Recalculado\",\"masa_kg_m\":20.25,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":25.8064,\"ix_cm4\":55.49752341333334,\"iy_cm4\":55.49752341333334,\"sx_cm3\":21.84941866666667,\"sy_cm3\":21.84941866666667,\"rx_cm\":1.4664696837416495,\"ry_cm\":1.4664696837416495,\"j_cm4\":93.79081456853335,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 0.472\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.128,\"d_ala_diametro_cm\":1.19888,\"bf_b_cm\":1.19888,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.4373132543999998,\"ix_cm4\":0.1721557826061599,\"iy_cm4\":0.1721557826061599,\"sx_cm3\":0.2871943524058453,\"sy_cm3\":0.2871943524058453,\"rx_cm\":0.34608684536302925,\"ry_cm\":0.34608684536302925,\"j_cm4\":0.2909432726044103,\"cw_cm6\":null}]"


## Bloque 4. Base de perfiles, orden y lista desplegable (perfiles_gerdau.py)

Obtener perfil por nombre y lista desplegable ordenada por familia y seccion.

In [ ]:
%%writefile perfiles_gerdau.py
"""Base de perfiles Gerdau rev04 (datos en gerdau_datos.py)."""
import json
from gerdau_datos import _DATOS_TEXTO

_DATOS = json.loads(_DATOS_TEXTO)
PERFILES = {p['perfil_comercial']: p for p in _DATOS}

FAMILIA_ORDEN = ['IR', 'CE', 'LI', 'SOL', 'OS', 'CS']


def obtener_perfil(nombre):
    """Dimensiones en cm: H = d (peralte o diametro), B = bf, e = tw, t = tf. None = dato pendiente."""
    p = PERFILES.get(nombre)
    if p is None:
        return None
    return {'familia': p['familia'], 'H': p['d_ala_diametro_cm'], 'B': p['bf_b_cm'],
            'e': p['tw_cm'], 't': p['tf_t_cm'], 'A': p['area_cm2'], 'Ix': p['ix_cm4'], 'Iy': p['iy_cm4'],
            'Sx': p['sx_cm3'], 'Sy': p['sy_cm3'], 'rx': p['rx_cm'], 'ry': p['ry_cm'], 'J': p['j_cm4'],
            'Cw': p['cw_cm6'], 'estado': p['estado'], 'masa': p['masa_kg_m']}


def _clave_seccion(p):
    """Familia, luego peralte (o diametro/lado), masa (mas liviano primero), ancho. Nulos al final."""
    def k(v):
        return (v is None, v or 0)
    return (FAMILIA_ORDEN.index(p['familia']),
            k(p['d_ala_diametro_cm']), k(p['masa_kg_m']), k(p['bf_b_cm']),
            p['perfil_comercial'])


def lista_perfiles_ordenada():
    """[(etiqueta, nombre)] para la lista desplegable. Muestra masa y estado si no es Revisado."""
    out = []
    for p in sorted(_DATOS, key=_clave_seccion):
        masa = '' if p['masa_kg_m'] is None else f"   {p['masa_kg_m']:g} kg/m"
        estado = '' if p['estado'] == 'Revisado' else f"   [{p['estado']}]"
        out.append((f"{p['perfil_comercial']}{masa}{estado}", p['perfil_comercial']))
    return out


## Bloque 5. Clics y modos de dibujo (dibujo_clics.py)

Modos Seleccionar, Nodo, Barra, Poli, Apoyo, Eje y Borrar, con ajuste a la rejilla.

In [ ]:
%%writefile dibujo_clics.py
"""
DIBUJO POR CLICS (estilo software de calculo estructural).
Motor de clics independiente de la interfaz: recibe coordenadas de un evento de matplotlib
(event.xdata, event.ydata, event.button) y modifica un dibujo_directo.DibujoModelo.

Vistas:
  - 'planta'   : plano X-Y a una cota Z (nivel actual)
  - 'alzado_y' : plano X-Z a una posicion Y fija
Modos:
  - 'nodo'   : cada clic crea un nodo
  - 'barra'  : dos clics = una barra (el 2do nodo queda como 'pendiente' hasta clic derecho)
  - 'poli'   : clics consecutivos encadenan barras; clic derecho termina
  - 'apoyo'  : clic sobre un nodo lo empotra
  - 'eje'    : clic 1 = seleccionar barra; clic 2 = direccion del eje fuerte
  - 'borrar' : clic sobre una barra la elimina
Ajuste (snap): cada clic se lleva a la interseccion de la rejilla mas cercana si esta dentro de
la tolerancia; si hay un nodo de la vista cerca, se usa ese nodo.
"""
import math
from dibujo_directo import DibujoModelo, ELEMENTOS

MODOS = ['seleccionar', 'nodo', 'barra', 'poli', 'apoyo', 'eje', 'borrar']


class MotorClics:
    def __init__(self, dib, xs, ys, zs, vista='planta', nivel=0.0, cota_y=0.0,
                 elemento='VIGA', perfil='IN30x36.7', modo='barra', tol_frac=0.25):
        self.dib = dib
        self.xs = sorted(xs); self.ys = sorted(ys); self.zs = sorted(zs)
        self.vista = vista; self.nivel = nivel; self.cota_y = cota_y
        self.elemento = elemento; self.perfil = perfil; self.modo = modo
        self.tol_frac = tol_frac
        self.pendiente = None          # indice de nodo de inicio (barra/poli)
        self.seleccion = None          # indice de barra (modo 'eje')
        self.log = []

    def _paso(self, lista):
        if len(lista) < 2:
            return 1.0
        return min(b - a for a, b in zip(lista[:-1], lista[1:]))

    def _ajustar(self, valor, rejilla):
        if not rejilla:
            return round(valor, 3)
        mas_cerca = min(rejilla, key=lambda r: abs(r - valor))
        if abs(mas_cerca - valor) <= self.tol_frac * self._paso(rejilla):
            return mas_cerca
        return round(valor, 3)

    def coordenada(self, xd, yd):
        """Convierte un clic (en unidades de la vista) a (x, y, z) ajustado a la rejilla."""
        if self.vista == 'planta':
            return (self._ajustar(xd, self.xs), self._ajustar(yd, self.ys), self.nivel)
        return (self._ajustar(xd, self.xs), self.cota_y, self._ajustar(yd, self.zs))

    def _en_vista(self, p):
        return abs(p[2] - self.nivel) < 1e-9 if self.vista == 'planta' else abs(p[1] - self.cota_y) < 1e-9

    def _nodo_en_vista(self, xd, yd):
        """Nodo visible cercano al clic (dentro de la tolerancia de la rejilla), o None."""
        mejor, dmin = None, None
        tol_h = self.tol_frac * self._paso(self.xs)
        tol_v = self.tol_frac * self._paso(self.ys if self.vista == 'planta' else self.zs)
        for i, q in enumerate(self.dib.nodos):
            if not self._en_vista(q):
                continue
            hx, vy = (q[0], q[1]) if self.vista == 'planta' else (q[0], q[2])
            if abs(hx - xd) <= tol_h and abs(vy - yd) <= tol_v:
                d = math.hypot(hx - xd, vy - yd)
                if dmin is None or d < dmin:
                    mejor, dmin = i, d
        return mejor

    def _nodo_de_clic(self, xd, yd):
        existente = self._nodo_en_vista(xd, yd)
        if existente is not None:
            return existente
        return self.dib.nodo(*self.coordenada(xd, yd))

    def _barra_cercana(self, xd, yd):
        """Indice de la barra visible mas cercana al clic (dentro de la tolerancia), o None."""
        mejor, dmin = None, None
        for i, b in enumerate(self.dib.barras):
            p1, p2 = self.dib.nodos[b['n1']], self.dib.nodos[b['n2']]
            if not (self._en_vista(p1) and self._en_vista(p2)):
                continue
            ax, ay = (p1[0], p1[1]) if self.vista == 'planta' else (p1[0], p1[2])
            bx, by = (p2[0], p2[1]) if self.vista == 'planta' else (p2[0], p2[2])
            dx, dy = bx - ax, by - ay
            L2 = dx * dx + dy * dy
            t = 0.0 if L2 == 0 else max(0.0, min(1.0, ((xd - ax) * dx + (yd - ay) * dy) / L2))
            d = math.hypot(ax + t * dx - xd, ay + t * dy - yd)
            if dmin is None or d < dmin:
                mejor, dmin = i, d
        tol = self.tol_frac * max(self._paso(self.xs), self._paso(self.ys if self.vista == 'planta' else self.zs))
        return mejor if (dmin is not None and dmin <= tol) else None

    def clic(self, xd, yd, boton=1):
        """Procesa un clic. boton 1 = izquierdo (accion); 3 = derecho (cancelar / terminar)."""
        if boton == 3:
            self.pendiente = None; self.seleccion = None
            self.log.append('cancelado / terminado')
            return self
        if self.modo == 'seleccionar':
            self.seleccion = self._barra_cercana(xd, yd)
            self.log.append(f'seleccionada barra {self.seleccion}' if self.seleccion is not None else 'sin barra')
        elif self.modo == 'nodo':
            self.log.append(f'nodo {self._nodo_de_clic(xd, yd)}')
        elif self.modo in ('barra', 'poli'):
            n = self._nodo_de_clic(xd, yd)
            if self.pendiente is None:
                self.pendiente = n
            elif n != self.pendiente:
                self.dib.barra(self.pendiente, n, self.elemento, self.perfil)
                self.log.append(f'barra {self.pendiente}-{n}')
                self.pendiente = n if self.modo == 'poli' else None
        elif self.modo == 'apoyo':
            n = self._nodo_en_vista(xd, yd)
            if n is not None:
                self.dib.apoyo(n)
                self.log.append(f'apoyo en nodo {n}')
        elif self.modo == 'borrar':
            i = self._barra_cercana(xd, yd)
            if i is not None:
                del self.dib.barras[i]
                self.log.append(f'borrada barra {i}')
        elif self.modo == 'eje':
            if self.seleccion is None:
                self.seleccion = self._barra_cercana(xd, yd)
            else:
                b = self.dib.barras[self.seleccion]
                p1, p2 = self.dib.nodos[b['n1']], self.dib.nodos[b['n2']]
                pm = [(p1[k] + p2[k]) / 2 for k in range(3)]
                # la direccion usa el clic SIN ajustar a la rejilla (es un vector, no un nodo)
                if self.vista == 'planta':
                    d = (xd - pm[0], yd - pm[1], 0.0)
                else:
                    d = (xd - pm[0], 0.0, yd - pm[2])
                self.dib.eje_fuerte(self.seleccion, d)
                self.log.append(f'eje fuerte barra {self.seleccion}')
                self.seleccion = None
        return self


## Bloque 6. Orientacion del eje fuerte / peralte (orientacion_gerdau.py)

Convencion del motor: el peralte es el eje local y. Vigas con peralte vertical, columnas con peralte en Y.

In [ ]:
%%writefile orientacion_gerdau.py
"""
ORIENTACION DEL EJE FUERTE (peralte) EN PANTALLA, con la base Gerdau Mexico rev04.

Convencion del motor (copiada de motor_analisis.py del otro chat, sin depender de ese paquete):
  xAxis = (p2 - p1)/L ; yAxis = vecxz x xAxis ; zAxis = xAxis x yAxis.
  El eje fuerte (Iz) es el eje local y => su direccion es el PERALTE d del perfil.

Defaults del motor (sin marca):
  - vigas/diagonales: peralte VERTICAL (eje fuerte resiste gravedad)
  - columnas:         peralte en Y global (vecxz = (1,0,0))
"""
import math
import numpy as np
import perfiles_gerdau as G


def vector_vecxz(p1, p2, elemento):
    """vecxz por defecto segun el tipo de elemento (igual que el motor)."""
    v = np.array(p2, float) - np.array(p1, float)
    L = np.linalg.norm(v)
    x = v / L
    if abs(x[2]) > 0.9:                       # columna (casi vertical)
        return (1.0, 0.0, 0.0)
    vx = np.cross(x, np.array([0.0, 0.0, 1.0]))   # viga/diagonal: yAxis vertical
    n = np.linalg.norm(vx)
    if n < 1e-6:
        return (1.0, 0.0, 0.0)
    vx = vx / n
    return (float(vx[0]), float(vx[1]), float(vx[2]))


def vecxz_desde_direccion(p1, p2, s):
    """vecxz que hace que el eje local y (= peralte) apunte segun s (proyectado perpendicular a la barra)."""
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    s = np.array(s, float)
    u = s - x * np.dot(s, x)
    nu = np.linalg.norm(u)
    if nu < 1e-6:
        return None
    u = u / nu
    v = np.cross(x, u)
    return (float(v[0]), float(v[1]), float(v[2]))


def vecxz_de_barra(nodos, b):
    """vecxz de una barra: usa 'eje_fuerte' si existe, si no el default por tipo de elemento."""
    p1 = nodos[b['n1']]; p2 = nodos[b['n2']]
    s = b.get('eje_fuerte')
    if s:
        v = vecxz_desde_direccion(p1, p2, s)
        if v is not None:
            return v
    return vector_vecxz(p1, p2, b.get('elemento'))


def ejes_locales(dib, i):
    """Ejes locales (x, y, z) de la barra i."""
    b = dib.barras[i]
    p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
    vecxz = np.array(vecxz_de_barra(dib.nodos, b), float)
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    y = np.cross(vecxz, x); y = y / np.linalg.norm(y)
    z = np.cross(x, y)
    return x, y, z


def dimensiones(perfil):
    """(d, bf) en metros desde Gerdau (H = d, B = bf en cm). None si el perfil no existe o no tiene dato."""
    p = G.obtener_perfil(perfil)
    if p is None or p['H'] is None or p['B'] is None:
        return None
    return float(p['H']) / 100.0, float(p['B']) / 100.0


def contorno(dib, i, escala=1.0):
    """4 vertices globales del rectangulo del perfil en el punto medio de la barra i.
    Alto = d a lo largo del eje local y (eje fuerte); ancho = bf a lo largo de z."""
    b = dib.barras[i]
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    pm = (p1 + p2) / 2
    dims = dimensiones(b['perfil'])
    if dims is None:
        return None
    d, bf = dims[0] * escala, dims[1] * escala
    x, y, z = ejes_locales(dib, i)
    c = [pm + sy * (d / 2) * y + sz * (bf / 2) * z for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]]
    return [tuple(float(v) for v in q) for q in c]


PRESETS = {'peralte vertical (Z)': (0.0, 0.0, 1.0),
           'peralte en X': (1.0, 0.0, 0.0),
           'peralte en Y': (0.0, 1.0, 0.0)}


def orientar(dib, i, preset):
    """Fija el peralte (eje fuerte) de la barra i con un preset o un vector. Rechaza direccion paralela a la barra."""
    b = dib.barras[i]
    s = np.array(PRESETS[preset] if isinstance(preset, str) else preset, float)
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    if abs(np.dot(s, x)) > 0.999:
        raise ValueError(f'el peralte no puede ser paralelo a la barra {i}; elija otra direccion')
    dib.eje_fuerte(i, tuple(s))
    return b['eje_fuerte']


def ortogonalidad(dib, i):
    """Angulo (grados) entre el eje fuerte asignado y la barra; 90 = perpendicular (correcto)."""
    b = dib.barras[i]
    if b['eje_fuerte'] is None:
        return None
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    s = np.array(b['eje_fuerte'], float)
    return math.degrees(math.acos(max(-1.0, min(1.0, abs(np.dot(s, x))))))


## Bloque 7. Interfaz de clics con planta, alzado y seccion (interfaz_clics.py)

Figura de planta o alzado, panel de informacion y seccion Gerdau de la barra seleccionada.

In [ ]:
%%writefile interfaz_clics.py
"""
INTERFAZ DE DIBUJO POR CLICS (matplotlib: funciona en Kaggle con %matplotlib widget / ipympl,
y en cualquier entorno para revisar la pantalla).

Layout:
  izquierda : modos (Nodo, Barra, Polilinea, Apoyo, Eje fuerte, Borrar), vista (Planta/Alzado),
              nivel Z, cota Y, perfil, elemento
  centro    : vista de dibujo con rejilla, nodos, barras por elemento, apoyos y ejes fuertes
  derecha   : resumen (nodos, barras, apoyos), modo activo, pendiente y ultimas acciones
"""
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, TextBox, RadioButtons
from dibujo_clics import MotorClics, MODOS
from dibujo_directo import ELEMENTOS
import numpy as np
import perfiles_gerdau as G


def _seccion_poligono(familia, H, B, tw, tf):
    """Contorno de la seccion en coordenadas (horizontal = eje debil, vertical = peralte)."""
    h, b = H / 2, B / 2
    if familia == 'CE':   # canal: alma a la izquierda, alas hacia +horizontal
        return [(-b, -h), (b, -h), (b, -h + tf), (-b + tw, -h + tf), (-b + tw, h - tf), (b, h - tf), (b, h), (-b, h)]
    if familia == 'IR':   # I laminada
        w = tw / 2
        return [(-b, -h), (b, -h), (b, -h + tf), (w, -h + tf), (w, h - tf), (b, h - tf), (b, h), (-b, h),
                (-b, h - tf), (-w, h - tf), (-w, -h + tf), (-b, -h + tf)]
    return None

COLOR = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}


def selector_perfil(interfaz, valor=None):
    """Lista desplegable de perfiles Gerdau, ordenada por familia y seccion.
    Se muestra encima de la figura; al elegir un perfil, las barras nuevas usan ese perfil."""
    import ipywidgets as W
    sel = W.Dropdown(options=G.lista_perfiles_ordenada(), value=valor or interfaz.motor.perfil,
                     description='Perfil:', layout=W.Layout(width='620px'))
    sel.observe(lambda cambio: interfaz.set_perfil(cambio['new']), names='value')
    return sel


class InterfazClics:
    def __init__(self, dib, xs, ys, zs, perfil='IR 12 × 16', elemento='VIGA', nivel=0.0, cota_y=0.0):
        self.dib = dib
        self.motor = MotorClics(dib, xs, ys, zs, vista='planta', nivel=nivel, cota_y=cota_y,
                                elemento=elemento, perfil=perfil, modo='barra')
        self.xs, self.ys, self.zs = sorted(xs), sorted(ys), sorted(zs)
        self.log = []
        self.fig = plt.figure(figsize=(14, 8))
        self.fig.canvas.manager.set_window_title('Dibujo por clics') if hasattr(self.fig.canvas, 'manager') and self.fig.canvas.manager else None
        self.ax = self.fig.add_axes([0.25, 0.08, 0.53, 0.86])
        self._construir_botones()
        self.info = self.fig.text(0.80, 0.97, '', va='top', family='monospace', fontsize=8)
        self.ax_sec = self.fig.add_axes([0.84, 0.04, 0.12, 0.18])
        self.ax_sec.set_visible(False)
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        self.redibujar()

    # ---------- botones y campos ----------
    def _construir_botones(self):
        self.btn = {}
        y0 = 0.93
        for i, m in enumerate(MODOS):
            ax_b = self.fig.add_axes([0.03, y0 - 0.045 * i, 0.17, 0.036])
            b = Button(ax_b, m.capitalize())
            b.on_clicked(lambda _e, m=m: self.set_modo(m))
            self.btn[m] = b
        ax_v = self.fig.add_axes([0.03, 0.62, 0.17, 0.036])
        self.btn_vista = Button(ax_v, 'Vista: planta')
        self.btn_vista.on_clicked(lambda _e: self.toggle_vista())
        ax_n = self.fig.add_axes([0.13, 0.54, 0.07, 0.03])
        self.tb_nivel = TextBox(ax_n, 'Nivel Z (m) ', initial=str(self.motor.nivel))
        self.tb_nivel.on_submit(self._set_nivel)
        ax_c = self.fig.add_axes([0.13, 0.48, 0.07, 0.03])
        self.tb_cota = TextBox(ax_c, 'Cota Y (m) ', initial=str(self.motor.cota_y))
        self.tb_cota.on_submit(self._set_cota)
        ax_e = self.fig.add_axes([0.03, 0.22, 0.17, 0.16])
        self.rb_elem = RadioButtons(ax_e, ELEMENTOS, active=ELEMENTOS.index(self.motor.elemento))
        self.rb_elem.on_clicked(self._set_elemento)
        ax_r = self.fig.add_axes([0.03, 0.10, 0.17, 0.04])
        self.btn_reset = Button(ax_r, 'Reiniciar modelo')
        self.btn_reset.on_clicked(lambda _e: self._reiniciar())

    # ---------- acciones ----------
    def set_modo(self, m):
        self.motor.modo = m; self.motor.pendiente = None; self.motor.seleccion = None
        self._registrar(f'modo: {m}'); self.redibujar()

    def toggle_vista(self):
        self.motor.vista = 'alzado_y' if self.motor.vista == 'planta' else 'planta'
        self.btn_vista.label.set_text('Vista: ' + ('alzado Y' if self.motor.vista == 'alzado_y' else 'planta'))
        self.motor.pendiente = None
        self._registrar(f'vista: {self.motor.vista}'); self.redibujar()

    def _set_nivel(self, txt):
        try:
            self.motor.nivel = float(txt.replace(',', '.'))
        except ValueError:
            return
        self._registrar(f'nivel Z = {self.motor.nivel}'); self.redibujar()

    def _set_cota(self, txt):
        try:
            self.motor.cota_y = float(txt.replace(',', '.'))
        except ValueError:
            return
        self._registrar(f'cota Y = {self.motor.cota_y}'); self.redibujar()

    def set_perfil(self, nombre):
        """Perfil con el que se crean las barras nuevas (lo fija la lista desplegable)."""
        self.motor.perfil = nombre; self._registrar(f'perfil = {nombre}'); self.redibujar()

    def _set_elemento(self, label):
        self.motor.elemento = label; self._registrar(f'elemento = {label}'); self.redibujar()

    def _reiniciar(self):
        self.dib.nodos.clear(); self.dib.barras.clear(); self.dib.apoyos.clear()
        self.motor.pendiente = None; self.motor.seleccion = None
        self._registrar('modelo reiniciado'); self.redibujar()

    def _registrar(self, texto):
        self.log.append(texto); self.log = self.log[-4:]

    # ---------- clics ----------
    def _on_click(self, event):
        if event.inaxes is not self.ax or event.xdata is None:
            return
        n_antes = len(self.motor.log)
        self.motor.clic(event.xdata, event.ydata, event.button)
        for linea in self.motor.log[n_antes:]:
            self._registrar(linea)
        self.redibujar()

    def simular_clic(self, xd, yd, boton=1):
        """Para pruebas: ejecuta el mismo camino que un clic real."""
        class _E: pass
        e = _E(); e.inaxes = self.ax; e.xdata = xd; e.ydata = yd; e.button = boton
        self._on_click(e)

    # ---------- dibujo ----------
    def _ejes_vista(self):
        if self.motor.vista == 'planta':
            return self.xs, self.ys, self.motor.nivel, 'X (m)', 'Y (m)'
        return self.xs, self.zs, self.motor.cota_y, 'X (m)', 'Z (m)'

    def redibujar(self):
        ax = self.ax; ax.cla()
        xs, rej_v, plano, lx, ly = self._ejes_vista()
        dib = self.dib; N = dib.nodos
        def en_plano(p):
            return abs(p[2] - plano) < 1e-9 if self.motor.vista == 'planta' else abs(p[1] - plano) < 1e-9
        def coord(p):
            return (p[0], p[1]) if self.motor.vista == 'planta' else (p[0], p[2])
        # rejilla
        for x in xs: ax.axvline(x, color='#bbb', lw=0.8, zorder=0)
        for v in rej_v: ax.axhline(v, color='#bbb', lw=0.8, zorder=0)
        ax.axhline(0, color='#eee', lw=0.5, zorder=0)
        # barras del plano
        for i, b in enumerate(dib.barras):
            p1, p2 = N[b['n1']], N[b['n2']]
            if not (en_plano(p1) and en_plano(p2)):
                continue
            (x1, y1), (x2, y2) = coord(p1), coord(p2)
            ax.plot([x1, x2], [y1, y2], color=COLOR[b['elemento']], lw=3.5, zorder=2, solid_capstyle='round')
            if b['eje_fuerte'] is not None:
                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                dx, dy = (b['eje_fuerte'][0], b['eje_fuerte'][1]) if self.motor.vista == 'planta' else (b['eje_fuerte'][0], b['eje_fuerte'][2])
                n = (dx * dx + dy * dy) ** 0.5
                if n > 1e-9:
                    s = 0.12 * max(max(xs) - min(xs), 1)
                    ax.annotate('', xy=(mx + s * dx / n, my + s * dy / n), xytext=(mx, my),
                                arrowprops=dict(arrowstyle='-|>', color='magenta', lw=2), zorder=4)
        # nodos del plano
        ids_plano = [i for i, p in enumerate(N) if en_plano(p)]
        for i in ids_plano:
            x, y = coord(N[i])
            ax.plot(x, y, 'o', color='k', ms=4, zorder=3)
            ax.text(x, y, f' {i}', fontsize=7, va='bottom', zorder=3)
        # apoyos
        for i in dib.apoyos:
            if i in ids_plano:
                x, y = coord(N[i]); ax.plot(x, y, 's', color='red', ms=9, mfc='none', mew=2, zorder=5)
        # pendiente
        if self.motor.pendiente is not None and self.motor.pendiente < len(N) and en_plano(N[self.motor.pendiente]):
            x, y = coord(N[self.motor.pendiente])
            ax.plot(x, y, 'o', mfc='none', mec='orange', ms=14, mew=2.5, zorder=6)
        ax.set_xlim(min(xs) - 1.5, max(xs) + 1.5)
        ax.set_ylim(min(rej_v) - 1.2, max(rej_v) + 1.2)
        ax.set_aspect('equal'); ax.set_xlabel(lx); ax.set_ylabel(ly)
        titulo = (f"Planta — nivel Z = {plano:g} m" if self.motor.vista == 'planta' else f"Alzado — cota Y = {plano:g} m")
        ax.set_title(titulo + f"   |   modo: {self.motor.modo}", fontsize=11)
        # barra seleccionada: resaltado y propiedades de la base Gerdau
        self.ax_sec.cla(); self.ax_sec.set_visible(False)
        det = ''
        sel = self.motor.seleccion
        if sel is not None and sel < len(dib.barras):
            b = dib.barras[sel]
            p1, p2 = N[b['n1']], N[b['n2']]
            (x1, y1), (x2, y2) = coord(p1), coord(p2)
            if en_plano(p1) and en_plano(p2):
                ax.plot([x1, x2], [y1, y2], color='k', lw=7, alpha=0.25, zorder=1)
            per = G.obtener_perfil(b['perfil'])
            if per is None:
                det = f"Barra {sel}: perfil '{b['perfil']}' no esta en la base Gerdau"
            else:
                det = (f"Barra {sel}  ({b['elemento']})\n{b['perfil']}\nfamilia: {per['familia']}\n"
                       f"d={per['H']:g} bf={per['B']:g} tw={per['e']:g} tf={per['t']:g} cm\n"
                       f"A={per['A']:g} cm2  Ix={per['Ix']:g}  Iy={per['Iy']:g} cm4\n"
                       f"eje fuerte: {'asignado' if b['eje_fuerte'] else 'por defecto del motor'}")
                pol = _seccion_poligono(per['familia'], per['H'], per['B'], per['e'], per['t'])
                if pol:
                    self.ax_sec.set_visible(True)
                    self.ax_sec.fill([q[0] for q in pol] + [pol[0][0]], [q[1] for q in pol] + [pol[0][1]],
                                     color='#9ecae1', edgecolor='k')
                    self.ax_sec.annotate('', xy=(0, per['H'] / 2 + 3), xytext=(0, 0),
                                         arrowprops=dict(arrowstyle='-|>', color='magenta', lw=2.5))
                    self.ax_sec.text(0.6, per['H'] / 2 + 1.5, 'eje fuerte', color='magenta', fontsize=7)
                    self.ax_sec.set_aspect('equal'); self.ax_sec.set_xticks([]); self.ax_sec.set_yticks([])
                    self.ax_sec.set_xlim(-per['B'] / 2 - 2, per['B'] / 2 + 3)
                    self.ax_sec.set_ylim(-per['H'] / 2 - 2, per['H'] / 2 + 5)
                    self.ax_sec.set_title('seccion', fontsize=8)
        # panel de informacion
        apoyos = len(dib.apoyos)
        txt = (f"Nodos: {len(N)}\nBarras: {len(dib.barras)}\nApoyos: {apoyos}\n\n"
               f"Modo: {self.motor.modo}\nPendiente: {self.motor.pendiente if self.motor.pendiente is not None else '-'}\n"
               f"Perfil: {self.motor.perfil}\nElemento: {self.motor.elemento}\n\n{det}\n\nUltimas acciones:\n" + '\n'.join('  ' + l for l in self.log))
        self.info.set_text(txt)
        self.fig.canvas.draw_idle()


## Bloque 8. Demo: marco, apoyos, orientacion, lista de perfiles y pantalla

Ejecutar al final. Muestra la figura interactiva con la lista de perfiles encima.

In [ ]:
# BLOQUE 8: demo
import matplotlib.pyplot as plt
from IPython.display import display
from dibujo_directo import DibujoModelo
from orientacion_gerdau import orientar, ortogonalidad, contorno
from interfaz_clics import InterfazClics, selector_perfil

xs, ys, zs = [0, 5, 10], [0, 4], [0, 3.5, 7]
dib = DibujoModelo(ruta='soportes_demo')
dib.marco_grilla(xs, ys, zs, 'IR 10 × 30', 'IR 12 × 16', 'IR 8 × 28')
for i, p in enumerate(dib.nodos):
    if abs(p[2]) < 1e-9:
        dib.apoyo(i)
print('nodos', len(dib.nodos), '| barras', len(dib.barras), '| apoyos', len(dib.apoyos))

# Peralte de una viga: vertical (Z) por defecto del motor; se fija explicitamente como ejemplo
i_viga = next(i for i, b in enumerate(dib.barras) if b['elemento'] == 'VIGA')
orientar(dib, i_viga, 'peralte vertical (Z)')
print('viga', i_viga, '| angulo eje fuerte vs barra =', round(ortogonalidad(dib, i_viga), 1), 'grados')
print('contorno de la seccion (4 vertices):', contorno(dib, i_viga) is not None)

modelo = dib.a_modelo()          # formato leer_dxf(): listo para el motor de analisis
ui = InterfazClics(dib, xs, ys, zs, perfil='IR 12 × 16', elemento='VIGA', nivel=3.5)
lista = selector_perfil(ui)       # lista desplegable de perfiles Gerdau (familia y seccion)
display(lista)
plt.show()
